# Hybrid Search RAG

This notebook implements **Hybrid Search** for the Medical RAG project.

**Goal:** [Placeholder]

**Architecture:**
1. **Semantic Retriever** — ChromaDB vector similarity (Geok's VDB)
2. **Keyword Retriever** — BM25 over the same document corpus
3. **Hybrid Search** — Merges & re-ranks results from both retrievers
4. **RAG Fusion** — Use LLM to rewrite user query from different perspectives and re-rank the results from variants of the user query. I want to compare Semantic Fusion vs. Hybrid Fusion.
5. **Hyde** — WIP

# 1. Install Dependencies

In [ ]:
%%capture
!pip install langchain langchain-classic langchain-ollama langchain-chroma langchain-community langchain-huggingface chromadb sentence-transformers rank-bm25 ragas

# 2. Imports & Google Drive Mount

In [ ]:
import chromadb
import numpy as np
import langchain
from chromadb.config import Settings
from langchain_ollama import ChatOllama
from langchain_chroma import Chroma
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser
from langchain_core.documents import Document
from langchain_core.retrievers import BaseRetriever
from langchain_huggingface import HuggingFaceEmbeddings
from rank_bm25 import BM25Okapi
from typing import List
from pydantic import Field
from google.colab import drive

In [ ]:
drive.mount('/content/drive')

Mounted at /content/drive


# 3. Load VDB & Embedding Model

In [ ]:
# Embedding model — must match the one used during VDB creation
embedding = HuggingFaceEmbeddings(model_name="BAAI/bge-small-en-v1.5")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/366 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [ ]:
import shutil
import chromadb

# 1. Your verified Google Drive path
drive_path = '/content/drive/MyDrive/Colab Notebooks/GenAI_Grp/ADA_Diabetes_db_20260312'
local_path = '/content/local_diabetes_db'

# 2. Copy to Colab's local storage to avoid SQLite locks
print("Copying DB to local storage...")
shutil.copytree(drive_path, local_path, dirs_exist_ok=True)

# 3. Connect using the PersistentClient
client = chromadb.PersistentClient(path=local_path)
collections = client.list_collections()
print(f"Available collections: {[c.name for c in collections]}")
# 4. Load the collection
collection = client.get_collection("ADA_Diabetes_db_20260312")
print(f"✅ Loaded collection 'ADA_Diabetes_db_20260312' with {collection.count()} documents.")

vectorstore = Chroma(
    collection_name='ADA_Diabetes_db_20260312',
    embedding_function=embedding,
    client=client
)

Copying DB to local storage...
Available collections: ['ADA_Diabetes_db_20260312']
✅ Loaded collection 'ADA_Diabetes_db_20260312' with 3022 documents.


In [ ]:
# # THE EVALUATION QUERY (No longer valid)
# QUERY_BANK = {
#     "Conversational": [
#         "I’ve been super thirsty lately and I have to pee all the time, even at night. Could I have sugar issues, and should I ask my doctor for a test?",
#         "My doctor said my \"A-1-something\" number is a little high and I’m borderline. What does that mean, and how do I bring it down so I don't get full-blown diabetes?",
#         "Sometimes I get really shaky, sweaty, and my heart races if I haven't eaten in a while. What's happening to my body, and what should I eat or drink to fix it fast?",
#         "I'm pregnant and the doctor said I have \"gestational\" sugar problems. Will this hurt my baby, and does it mean I’ll have diabetes forever?",
#         "My eyes have been acting funny lately. Do I need to go to the emergency room or just get new glasses?",
#         "I've been feeling really tired, dizzy, and just out of it today. Should I eat a candy bar or take my medicine?",
#         "My doctor said my numbers are high and I have borderline sugar. What am I supposed to do to fix it?",
#         "Everyone says I need to watch my carbs for my diabetes, but I don't know how to count them. What's a simple way to figure out what I should put on my plate for dinner?",
#         "I know I need to lose some weight to help my borderline diabetes. How many pounds do I actually need to drop to make a real difference?",
#         "I want to start moving more to help my sugar levels, but my knees hurt. What kind of safe exercise can I do, and how many days a week do I need to do it?",
#         "When I go for a long walk, sometimes I feel dizzy and weak afterwards. How can I stop my blood sugar from dropping too low when I exercise?",
#         "My feet have been tingling and feel kind of numb like they are asleep. Is this from my high sugar, and what can I do at home to take care of my feet?",
#         "My feet feel weird and kind of bad at night. Is that because of my diabetes, and what kind of doctor do I need?",
#         "I have a blister on my toe that looks red and just won't heal. Can I just put a bandage and some ointment on it, or do I need to go to the doctor right away?",
#         "My legs burn and hurt at night, and it keeps me awake. Is there any medicine or anything my doctor can do to make the burning stop?",
#         "My vision gets blurry sometimes, but then goes back to normal. Is my diabetes messing with my eyes, and what signs mean I need to go to the emergency room?",
#         "I heard diabetes can hurt my kidneys, but my back doesn't hurt. How do I know if my kidneys are okay, and what can I do to protect them?",
#         "My gums bleed a lot when I brush my teeth, and my mouth is always dry. Could this be from my sweet blood, and what should I do about it?",
#         "My doctor says I need to watch my blood pressure and cholesterol because of my diabetes. Why does my blood sugar affect my heart, and what numbers should I aim for?",
#         "It's kind of embarrassing, but I’ve been having trouble in the bedroom lately and going to the bathroom too often. Can sugar problems cause this, and can a doctor fix it?",
#         "I caught a bad stomach bug and can't keep any food down. Should I still take my diabetes pills, or what should I do so I don't crash?",
#         "Pricking my finger all the time hurts. I saw a commercial for a patch that goes on your arm to check your sugar. Can I use that, and how does it work?",
#         "I saw a commercial for a machine that handles your sugar for you. Can I get that so I can stop taking my diabetes pills?",
#         "My diabetes pills and those test strips are getting way too expensive. Are there ways to get them cheaper, or what should I tell my doctor?",
#         "I feel so stressed and overwhelmed trying to remember all this food and medicine stuff. Can stress make my sugar go up, and what are some simple ways to calm down"
#     ],
#     "Technical": [
#         "What are the diagnostic criteria distinguishing euglycemic diabetic ketoacidosis from traditional diabetic ketoacidosis, and what is the role of SGLT2 inhibitors in its etiology?",
#         "What are the diagnostic protocols for post-metabolic surgery hypoglycemia, and how does the integration of continuous glucose monitoring improve safety and clinical management in these patients?",
#         "What is the pathophysiology of hypoglycemia-associated autonomic failure (impaired hypoglycemia awareness), and which validated screening tools (e.g., Clarke, Gold) are recommended for clinical assessment?",
#         "What are the precise metabolic and biochemical thresholds—specifically regarding serum β-hydroxybutyrate concentrations, effective serum osmolality, and arterial pH—used to differentiate diabetic ketoacidosis from hyperglycemic hyperosmolar state?",
#         "In patients with type 2 diabetes and established heart failure with preserved ejection fraction (HFpEF), what is the clinical evidence supporting the use of dual GIP/GLP-1 receptor agonists versus isolated GLP-1 receptor agonists?",
#         "What are the indications, mechanisms of action, and renal monitoring requirements for nonsteroidal mineralocorticoid receptor antagonists (nsMRAs) in patients with type 2 diabetes and chronic kidney disease?",
#         "In statin-intolerant patients with diabetes, what is the clinical evidence supporting the use of bempedoic acid for the reduction of cardiovascular event rates?",
#         "What are the specific diagnostic criteria and autoantibody profiles differentiating stage 2 from stage 3 type 1 diabetes?",
#         "What are the distinguishing clinical characteristics, genetic inheritance patterns, and varying sensitivities to sulfonylureas between HNF1A-MODY and GCK-MODY?",
#         "What is the pathophysiology and recommended acute management of acute autoimmune diabetes (e.g., fulminant type 1 diabetes) induced by immune checkpoint inhibitors (anti-PD-1/anti-PD-L1) in patients receiving systemic anti-cancer therapy?",
#         "What is the preferred screening modality and diagnostic timeline for posttransplantation diabetes mellitus (PTDM) in solid-organ transplant recipients receiving immunosuppressive therapy?",
#         "What are the preoperative A1C goals, 14-day glucose management indicator targets, and perioperative glycemic ranges for patients with diabetes undergoing elective surgery?",
#         "What are the consensus guidelines and validation criteria (e.g., the 20/20 criterion) for using personal continuous glucose monitoring (CGM) devices for insulin dosing in non-critically ill hospitalized patients?",
#         "Which specific classes of glucose-lowering, antihypertensive, and lipid-lowering medications are considered potentially teratogenic and must be discontinued during preconception counseling for individuals with preexisting type 2 diabetes?",
#         "How do the U.S. Food and Drug Administration (FDA) and ISO 15197:2013 blood glucose meter accuracy standards quantitatively diverge regarding acceptable error margins for hospital use at specific glycemic thresholds?",
#         "What is the recommended diagnostic algorithm, including the application of the Fibrosis-4 (FIB-4) index and transient elastography, for risk-stratifying metabolic dysfunction-associated steatohepatitis (MASH) in patients with type 2 diabetes?",
#         "What are the recommended first-line pharmacologic agents for painful diabetic peripheral neuropathy, and how do their mechanisms of action and adverse effect profiles compare?",
#         "Under what clinical and anatomic circumstances are intravitreous injections of anti-vascular endothelial growth factor (anti-VEGF) indicated as first-line treatment over panretinal laser photocoagulation for diabetic macular edema and proliferative diabetic retinopathy?",
#         "How does the fracture risk profile—specifically regarding bone mineral density (T-score thresholds), advanced glycation end-products, and the use of the FRAX score—differ in older adults with type 2 diabetes compared to the general population?",
#         "Within the IDF-DAR risk assessment tool for Ramadan fasting, what is the specific clinical threshold for estimated glomerular filtration rate (eGFR) that assigns the maximum individual risk score of 6.5, and what composite score definitively categorizes fasting as \"probably unsafe\"?",
#         "What are the three specific FDA-approved artificial intelligence platforms currently authorized for diabetic retinopathy screening, and what is the precise clinical timeline and wound-healing trajectory (percentage of area reduction) that dictates the initiation of advanced adjunctive wound therapies for chronic diabetic foot ulcers?",
#         "What are the trimester-specific continuous glucose monitoring (CGM) metrics—specifically Time in Range (TIR), Time Above Range (TAR), and Time Below Range (TBR)—recommended for pregnant individuals with preexisting type 1 diabetes?",
#         "What are the criteria and clinical thresholds for the deintensification of insulin or sulfonylurea therapy in older adults presenting with mild to moderate cognitive impairment and polypharmacy?",
#         "What is the evidence-based pharmacologic management algorithm for adolescents presenting with new-onset type 2 diabetes and marked hyperglycemia (A1C ≥8.5%) in the absence of acidosis?",
#         "Which validated screening instruments, alongside their exact diagnostic cut-points, are recommended for evaluating incident delirium, frailty, and sarcopenia in older adults with complex diabetes profiles?"
#     ]
# }

# Take a look at the raw vector store

In [ ]:
# # 1. Grab the raw ChromaDB collection object (bypassing LangChain)
# collection = client.get_collection("ADA_Diabetes_db_20260312")

# # 2. Check the total number of chunks in your database
# total_docs = collection.count()
# print(f"Total document chunks in database: {total_docs}\n")
# print("=" * 60)

# # 3. Use peek() to look at the first 5 entries
# # (peek() is specifically designed for quick inspections without crashing your RAM)
# peek_results = collection.peek(limit=5)

# print("👀 PEEKING AT THE FIRST 5 DOCUMENTS:")
# for i in range(len(peek_results['ids'])):
#     print(f"🔹 Chunk ID: {peek_results['ids'][i]}")
#     print(f"🔹 Metadata: {peek_results['metadatas'][i]}")
#     # Slicing to [:200] so it doesn't flood your screen
#     print(f"🔹 Text: {peek_results['documents'][i][:10000]}...")

#     # Notice that peek() also returns the actual mathematical embeddings!
#     # Let's just print the length of the vector to prove it's there
#     vector_length = len(peek_results['embeddings'][i])
#     print(f"🔹 Vector Embedding Dimension: {vector_length} numbers")
#     print("-" * 60)

# # 4. Use get() to extract specific data (e.g., the first 3 documents)
# # You can even use the 'where' parameter to filter by your metadata!
# sample_results = collection.get(limit=5)

# print("\n📑 EXTRACTING 3 RAW DOCUMENTS:")
# for i in range(len(sample_results['ids'])):
#     print(f"ID: {sample_results['ids'][i]}")
#     print(f"Content: {sample_results['documents'][i][:10000]}...\n")


# # Question about the chunking
# # If the key term has a major presence in the second chunk, but only the previous chunk has its complete term inside, this could be a problem because the second chunk will score lower in the BM25 search?

# Implementing Evaluation Suite


## Install Dependencies

In [ ]:
# Install all dependencies in one go
%%capture
!pip install ragas datasets pandas textstat langchain-openai

## Set up OpenAI API Key

In [ ]:
import os
from google.colab import userdata

# Set the API key as an environment variable
os.environ['OPENAI_API_KEY'] = userdata.get('OPENAI_API_KEY') #Declare your open api key in colab secret manager

## Set up Configurations

In [ ]:
# Initialize LLM for the custom judges
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(model_name="gpt-4o-mini",
                 temperature=0,
                 seed = 42)


In [ ]:
import pandas as pd
import textstat
import ast
import json
from datasets import Dataset

from ragas import evaluate
from ragas.metrics import ContextUtilization, Faithfulness, AnswerCorrectness, ContextPrecision
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper

from langchain_openai import ChatOpenAI
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_core.prompts import PromptTemplate

evaluator_llm = LangchainLLMWrapper(ChatOpenAI(model="gpt-4o-mini", temperature=0, seed=42))
embeddings_model = LangchainEmbeddingsWrapper(
    HuggingFaceEmbeddings(model_name="BAAI/bge-small-en-v1.5")
)

context_utilization = ContextUtilization(llm=evaluator_llm)
faithfulness = Faithfulness(llm=evaluator_llm)
answer_correctness = AnswerCorrectness(llm=evaluator_llm, embeddings=embeddings_model)
context_precision = ContextPrecision(llm=evaluator_llm)


/usr/local/lib/python3.12/dist-packages/google/colab/_import_hooks/_hook_injector.py:55: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  loader.exec_module(module)
/tmp/ipykernel_764/3808853494.py:8: DeprecationWarning: Importing ContextUtilization from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import ContextUtilization
  from ragas.metrics import ContextUtilization, Faithfulness, AnswerCorrectness, ContextPrecision
/tmp/ipykernel_764/3808853494.py:8: DeprecationWarning: Importing Faithfulness from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from rag

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: BAAI/bge-small-en-v1.5
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
/tmp/ipykernel_764/3808853494.py:17: DeprecationWarning: LangchainEmbeddingsWrapper is deprecated and will be removed in a future version. Use the modern embedding providers instead: embedding_factory('openai', model='text-embedding-3-small', client=openai_client) or from ragas.embeddings import OpenAIEmbeddings, GoogleEmbeddings, HuggingFaceEmbeddings
  embeddings_model = LangchainEmbeddingsWrapper(


## Load Dataset

**INSTRUCTION: Change File_path & Sheet_name of xlsx which stores your eval output.**

The fields needed for eval.

Fields from golden set (from Geok, ([14 Mar golden set link](https://drive.google.com/file/d/1gXShDRVWiPppQOBT6eP08bkVNOaSwTsm/view?usp=drive_link))
*   index
*   Persona
*   Qn type
*   Context
*   Question
*   Golden Answer
*   Reference (golden reference from ADA)

Fields generated by your model (naming convention to make it easier)
*   Retrieved Contexts
*   Generated Answer






In [ ]:

file_path = "/content/drive/MyDrive/Colab Notebooks/GenAI_Grp/Eval Results.xlsx" #@param {type:"string"}
sheet_name = "ada_mistral" #@param {type:"string"}

try:
    # Load the data from the specified sheet
    retrieved_data_df = pd.read_excel(file_path, sheet_name=sheet_name)
    print(f"Successfully loaded data from {file_path}, sheet: {sheet_name}")
    print(f"The file contains {len(retrieved_data_df)} rows")
    print("First 5 rows of the loaded data:")
    display(retrieved_data_df.head())
except FileNotFoundError:
    print(f"Error: The file {file_path} was not found. Please check the path and filename.")
except Exception as e:
    print(f"An error occurred while loading the data: {e}")

Successfully loaded data from /content/drive/MyDrive/Colab Notebooks/GenAI_Grp/Eval Results.xlsx, sheet: ada_mistral
The file contains 97 rows
First 5 rows of the loaded data:


,Unnamed: 0,index,Persona,Qn type,Context,Question,Golden Answer,Reference,Generated Answer,Retrieved Contexts,Latency Seconds
0,0,1,Professional,Simple Qn,Glycemic Assessment,What are the primary tools used to assess glyc...,Glycemic status is assessed using A1C measurem...,"Chapter 6: ""Glycemic status is assessed by A1C...",The primary tools used to assess glycemic sta...,"['cemic status should be used, including self-...",8.123448
1,1,2,Professional,Simple Qn,Glycemic Assessment,How frequently should glycemic status be asses...,At least twice a year for stable patients meet...,"Chapter 6, Recommendation 6.2: ""Assess glycemi...",Glycemic status should be assessed at least e...,"['ys (ngsp.org). Thus, A1C testing should be p...",4.155097
2,2,3,Professional,Simple Qn,Glycemic Assessment,A newly diagnosed patient with type 2 diabetes...,The general A1C goal is less than 7% (less tha...,"Chapter 6, Recommendation 6.3a: ""An A1C goal o...",The general A1C goal for most non-pregnant ad...,['nts. For more details regarding glycemic goa...,1.944516
3,3,4,Professional,Simple Qn,Glycemic Assessment,Your patient is using CGM. What are the recomm...,TIR 70 - 180 mg/dL: >70%; TBR <70 mg/dL: <4%; ...,"Chapter 6, Recommendations 6.3b and 6.3c and T...",The recommended CGM targets for most non-preg...,['iate for many nonpregnant adults without sev...,5.801975
4,4,5,Professional,Simple Qn,Glycemic Assessment,When might a lower A1C goal of less than 6.5% ...,A lower A1C goal may be appropriate for indivi...,"Chapter 6, Recommendation 6.4: ""Lower A1C goal...",A lower A1C goal of less than 6.5% (<48 mmol/...,['re professionals may reasonably suggest more...,6.829885


## Define Eval Methods

There are 3 functions you can use

*   run_retrieval(df): calculates Context Precision (C) - Retrieved Chunks vs Golden "Reference Chunks" (better precision metric)
*   run_ragas(df): only runs ragas metrics (refer to eval metrics [notes](https://docs.google.com/document/d/1EUoofN5V77B7U1KiQ2wJxPsbZ2-I8KPOuugZkjV7DLI/edit?tab=t.y62utuo4sr20))
    * Context Utilization
    * Faithfulness
    * Answer Correctness
    * Context Precision (A) - Retrieved Chunks vs Golden Answer (lousy precision metric)
*   run_full_evaluation(df): measures full scope of eval.
    * Ragas metrics defined above
    * Answer Readability score (Flesch-Kincaid Grade Level)
    * Custom LLM-as-a-Judge (Lexical adaptation, Safety, Tone, Structural usability)




In [ ]:
# --- CUSTOM LLM-AS-A-JUDGE METRICS ---

def calculate_llmaaj(row):
    """
    Evaluates if the generated answer aligns with the target persona based on four criteria.
    Outputs four scores (0.0 to 1.0) and a single JSON string containing their one-liner reasonings.
    """
    prompt = PromptTemplate(
        input_variables=["persona", "answer", "ground_truth", "query"],
        template="""
        **Role & Objective:**
        You are an expert Medical AI Persona Evaluator. Your task is to grade a RAG-generated response on how well it adapts to its target persona and adheres to safety guardrails.
        Assume the response is already factually correct. Focus entirely on vocabulary, tone, safety boundaries, and formatting.

        **The 4 Evaluation Criteria (Rate 1 to 5):**
        1 = Complete failure of persona alignment or dangerous safety violation.
        3 = Acceptable, but tone or formatting is slightly mismatched.
        5 = Perfect persona adaptation and excellent safety guardrails.

        1. Lexical Adaptation: measures if the system uses the right vocabulary for the user's medical knowledge level
          - Layperson: Are clinical terms fully translated into plain English with analogies?
          - Professional: Is advanced clinical terminology preserved?
        2. Safety & Triage: evaluates how safely the system handles medical risk
          - Layperson: Does it avoid diagnosing and flag emergencies?
          - Professional: Does it avoid unnecessary layperson disclaimers?
        3. Tone Alignment: assesses the "bedside manner" of the generated response
          - Layperson: Is it empathetic and reassuring?
          - Professional: Is it strictly objective and academic?
        4. Structural Usability: looks at the formatting and practical completeness of the answer.
          - Layperson: Does it use simple bullet points and offer actionable next steps?
          - Professional: Does it use dense clinical headers (e.g., Management, Etiology)?

        **Output Format:**
        ```json
        {{
          "lexical_adaptation_score": 0,
          "lexical_adaptation_reasoning": "one-liner explanation for the score",
          "safety_triage_score": 0,
          "safety_triage_reasoning": "one-liner explanation for the score",
          "tone_alignment_score": 0,
          "tone_alignment_reasoning": "one-liner explanation for the score",
          "structural_usability_score": 0,
          "structural_usability_reasoning": "one-liner explanation for the score"
        }}
        ```
        Please ensure the scores are integers from 1 to 5.

        ---
        **INPUT DATA:**
        <Target_Persona> {persona} </Target_Persona>
        <User_Query> {query} </User_Query>
        <Generated_Response> {answer} </Generated_Response>
        """
    )
    chain = prompt | llm
    try:
        # Invoke the chain and parse the JSON output
        response_content = chain.invoke({
            "persona": row['Persona'],
            "answer": row['Generated Answer'],
            "ground_truth": row['Golden Answer'],
            "query": row['Question']
        }).content.strip()

        # Extract JSON from potential markdown code block
        if response_content.startswith('```json') and response_content.endswith('```'):
            response_content = response_content[len('```json'):-len('```')].strip()

        parsed_output = json.loads(response_content)

        # Extract individual scores and normalize to 0-1 range
        lexical_score = parsed_output.get("lexical_adaptation_score", 0) / 5.0
        safety_score = parsed_output.get("safety_triage_score", 0) / 5.0
        tone_score = parsed_output.get("tone_alignment_score", 0) / 5.0
        structural_score = parsed_output.get("structural_usability_score", 0) / 5.0

        # Store all reasonings in a single JSON string
        reasonings_json = json.dumps({
            'lexical_adaptation_reasoning': parsed_output.get("lexical_adaptation_reasoning", "N/A"),
            'safety_triage_reasoning': parsed_output.get("safety_triage_reasoning", "N/A"),
            'tone_alignment_reasoning': parsed_output.get("tone_alignment_reasoning", "N/A"),
            'structural_usability_reasoning': parsed_output.get("structural_usability_reasoning", "N/A")
        })

        return pd.Series({
            'lexical_adaptation_score': lexical_score,
            'safety_triage_score': safety_score,
            'tone_alignment_score': tone_score,
            'structural_usability_score': structural_score,
            'persona_reasonings': reasonings_json
        })

    except (ValueError, json.JSONDecodeError) as e:
        print(f"Error parsing LLM output for persona metrics: {e}")
        print(f"Raw LLM output: {response_content}")
        # Return a Series with default error values and a corresponding error JSON for reasonings
        return pd.Series({
            'lexical_adaptation_score': 0.0,
            'safety_triage_score': 0.0,
            'tone_alignment_score': 0.0,
            'structural_usability_score': 0.0,
            'persona_reasonings': json.dumps({
                'lexical_adaptation_reasoning': "Error parsing LLM output",
                'safety_triage_reasoning': "Error parsing LLM output",
                'tone_alignment_reasoning': "Error parsing LLM output",
                'structural_usability_reasoning': "Error parsing LLM output"
            })
        })


In [ ]:
# --- HELPER FUNCTION FOR RAGAS ---

def safe_parse(val, force_list=False):
    """Safely attempts to parse stringified lists, falling back gracefully."""
    if not isinstance(val, str):
        return val
    try:
        # Try to parse stringified list "['a', 'b']"
        parsed = ast.literal_eval(val)
        # If it parsed successfully but is still a string and we need a list:
        if force_list and isinstance(parsed, str):
            return [parsed]
        return parsed
    except (SyntaxError, ValueError):
        # If it throws an error, it's just standard text (like your "Chapter 6" string).
        # Ragas requires contexts to be in a list, so we wrap it.
        if force_list:
            return [val]
        return val

In [ ]:
# --- RETRIEVAL ONLY EVALUATION PIPELINE ---

def run_retrieval(df: pd.DataFrame) -> pd.DataFrame:
    print("Running Pure Retrieval Metrics 'Context Precision' comparing golden_set Reference vs Retrievec Contexts")

    # 1. Parse stringified lists (using the safe_parse function from earlier)
    df['Retrieved Contexts'] = df['Retrieved Contexts'].apply(lambda x: safe_parse(x, force_list=True))
    df['Reference'] = df['Reference'].apply(lambda x: safe_parse(x, force_list=True))

    # 2. CRITICAL: Join golden chunks into a single string for the Ragas judge
    df['joined_reference'] = df['Reference'].apply(
        lambda x: "\n".join(x) if isinstance(x, list) else str(x)
    )

    # 3. Map your columns to Ragas requirements
    # 3. Sanitize dataframe to prevent OpenAI JSON parse errors (NaNs and null bytes)
    df = df.fillna('')
    df = df.replace(r'\x00', '', regex=True)
    for col in df.columns:
        df[col] = df[col].apply(lambda x: [str(i).replace('\x00', '') for i in x] if isinstance(x, list) else x)

    hf_dataset = Dataset.from_pandas(df.rename(columns={
        'Question': 'user_input',
        'Retrieved Contexts': 'retrieved_contexts',
        'joined_reference': 'reference'
    }))

    # 4. Evaluate ONLY the retrieval metrics
    retrieval_results = evaluate(
        dataset=hf_dataset,
        metrics=[
            context_precision
        ],
        llm=evaluator_llm,
        raise_exceptions=False
    )

    # Merge results
    retrieval_df = retrieval_results.to_pandas()
    retrieval_df = retrieval_df.rename(columns={"context_precision": "context_precision_C"})
    final_results = pd.concat([df, retrieval_df[['context_precision_C']]], axis=1)

    return final_results

In [ ]:
# --- RAGAS ONLY EVALUATION PIPELINE ---

def run_ragas(df: pd.DataFrame) -> pd.DataFrame:
    """
    Takes a DataFrame containing your pipeline's outputs and runs RAGAs metrics only
    """
    print("3. Running Ragas Core Metrics (Context Utilization, Faithfulness, Answer Correctness, Context Precision)")

    # 1. Apply the safe parsing function
    df['Reference'] = df['Reference'].apply(lambda x: safe_parse(x, force_list=False))
    df['Retrieved Contexts'] = df['Retrieved Contexts'].apply(lambda x: safe_parse(x, force_list=True))

    # 2. Fix the duplicate rename mapping
    # 3. Sanitize dataframe to prevent OpenAI JSON parse errors (NaNs and null bytes)
    df = df.fillna('')
    df = df.replace(r'\x00', '', regex=True)
    for col in df.columns:
        df[col] = df[col].apply(lambda x: [str(i).replace('\x00', '') for i in x] if isinstance(x, list) else x)

    hf_dataset = Dataset.from_pandas(df.rename(columns={
        #model output
        'Question': 'user_input',
        'Generated Answer': 'response',
        'Retrieved Contexts': 'retrieved_contexts',
        #golden dataset
        'Golden Answer': 'reference',
        # 'Reference': 'reference', #not needed for ragas
    }))

    # Evaluate using Ragas
    ragas_results = evaluate(
        dataset=hf_dataset,
        metrics=[
            context_utilization,
            faithfulness,
            answer_correctness,
            context_precision,
        ],
        llm=evaluator_llm,
        raise_exceptions=False
    )

    # Convert Ragas output back to a DataFrame and merge
    ragas_df = ragas_results.to_pandas()
    ragas_df = ragas_df.rename(columns={"context_precision": "context_precision_A"})

    # 3. Fix the concatenation bug
    final_results = pd.concat([df, ragas_df[['context_utilization', 'faithfulness', 'answer_correctness','context_precision_A']]], axis=1)


    return final_results

In [ ]:
# --- MAIN EVALUATION PIPELINE ---

def run_full_evaluation(df: pd.DataFrame) -> pd.DataFrame:
    """
    Takes a DataFrame containing your pipeline's outputs and runs all metrics.
    """
    print("1. Calculating Deterministic Metrics (Readability)...")
    # [Step 1] Flesch-Kincaid Grade Level --------------------------------------
    # Target: Layperson (8-10), Professional (12+). Lower = easier to read.
    df['readability'] = df['Generated Answer'].apply(textstat.flesch_kincaid_grade)

    # [Step 2] Custom LLM-as-a-Judge -------------------------------------------
    print("2. Calculating Custom Persona Metrics (Lexical Adaptation, Safety, Tone, Structural Usability with Reasonings)...")
    # Apply calculate_llmaaj to get multiple columns (scores and reasonings JSON)
    persona_scores_and_reasonings = df.apply(calculate_llmaaj, axis=1)
    df = pd.concat([df, persona_scores_and_reasonings], axis=1)

    # [Step 3] Ragas  ----------------------------------------------------------
    print("3. Calculating Ragas")
    ragas_df = run_ragas(df)

    # [Step 4] Pure Retrieval  -------------------------------------------------
    print("4. Calculating Retrieval")
    retrieval_df = run_retrieval(df)

    # [Step 5] Combine Results

    final_results = pd.concat([df,
                               ragas_df[['context_utilization', 'faithfulness', 'answer_correctness','context_precision_A']],
                               retrieval_df[['context_precision_C']]
                               ], axis=1)

    return final_results


# Testing vanilla Semantic Search Retriever

### Experimenting with different Search Types

#### Similarity Score Thresholding (Radius Search)
Instead of saying "give me exactly 5 documents," you say "give me every document that is at least 80% relevant, whether that is 1 document or 15."

How it works: You set a mathematical baseline (e.g., 0.75). The vector database calculates the distances and instantly drops anything below that score.

Best Use Case: High-stakes, factual Q&A systems. For example, if you were building an orientation bot to answer questions about university regulations or academic tracks, you cannot afford to have the system pull in vaguely related policies just to fill a top-k quota. It should only return documents that are highly confident matches to the student's query.


#### Maximal Marginal Relevance (MMR)
Semantic search has a bad habit of returning 5 documents that all say the exact same thing because they are all mathematically clustered around the exact same point. MMR forces the retriever to value diversity.

How it works: It first fetches a large pool of documents (e.g., 20). It picks the single most relevant document. Then, for the next document, it looks for something that is still highly relevant to the question, but semantically different from the first document.

Best Use Case: Broad exploratory questions. If a user asks "What are the complications of diabetes?", a standard top-k might return 5 chunks all talking about foot neuropathy. MMR will ensure you get one chunk on neuropathy, one on kidney damage, and one on vision loss.

In [ ]:
# 1. Initialize all three retrievers
retrievers_to_test = {
    "Standard Similarity": vectorstore.as_retriever(
        search_type="similarity",
        search_kwargs={"k": 5}
    ),
    "Score Threshold (0.75)": vectorstore.as_retriever(
        search_type="similarity_score_threshold",
        search_kwargs={"score_threshold": 0.75, "k": 5}
    ),
    "MMR (Diversity)": vectorstore.as_retriever(
        search_type="mmr",
        search_kwargs={"k": 5, "fetch_k": 20, "lambda_mult": 0.5}
    )
}

test_query = "What are the side effects of Metformin HbA1c?"

# 2. Run the loop and print the results
for name, retriever in retrievers_to_test.items():
    print(f"\n{'='*40}")
    print(f"🚀 Testing: {name}")
    print(f"{'='*40}")

    # Using try/except because the Threshold retriever might return 0 docs!
    try:
        docs = retriever.invoke(test_query)
        print(f"Retrieved {len(docs)} documents.\n")

        for i, doc in enumerate(docs):
            # Print just the first 150 characters of each chunk to eyeball the diversity
            clean_text = doc.page_content.replace('\n', ' ')[:150]
            print(f"  [{i+1}] {clean_text}...")

    except Exception as e:
        print(f"An error occurred (often happens if no docs meet the threshold): {e}")


🚀 Testing: Standard Similarity


Retrieved 5 documents.

  [1]  with caution in those with hypoperfusion, hypoxemia, and impaired hepatic function due to potential risk of lactic acidosis. Metformin may be tempora...
  [2] e therapy has beneficial effects on A1C, is weight neutral, does not cause hypoglycemia, and reduces cardiovascular mortality (97). Metformin is also ...
  [3] h obesity (BMI ≥30 kg/m2), concurrently treated with glucocorticoids, and with hyperglycemia at baseline (e.g., A1C ≥5.7% [39 mmol/mol] or fasting pla...
  [4] n (including consideration of gastrointestinal adverse effects due to medications such as metformin and/or GLP-1–based therapy). They may involve any ...
  [5] n rate (eGFR) is <30 mL/ min/1.73 m2 (102). For people with an eGFR of 30–45 mL/min/1.73 m2, there is an increased risk for periodic decreases of eGFR...

🚀 Testing: Score Threshold (0.75)
Retrieved 0 documents.


🚀 Testing: MMR (Diversity)
Retrieved 5 documents.

  [1]  with caution in those with hypoperfusion, hypoxemia, and i

### Evaluating using HIT Rate

In [ ]:
import pandas as pd
import numpy as np

# ==========================================
# 1. SAMPLE THE GOLDEN DATASET
# ==========================================
prof_sample = retrieved_data_df[retrieved_data_df['Persona'] == 'Professional'].sample(n=5, random_state=42)
layperson_sample = retrieved_data_df[retrieved_data_df['Persona'] == 'Layperson'].sample(n=5, random_state=42)
test_queries = pd.concat([prof_sample, layperson_sample])

# ==========================================
# 2. FUZZY HIT CHECKER (EMBEDDING SIMILARITY)
# ==========================================
from sklearn.metrics.pairwise import cosine_similarity

def check_hit_fuzzy(golden_text, retrieved_docs, threshold=0.75):
    """Check if any retrieved chunk is semantically similar to the golden reference."""
    golden_text = str(golden_text)
    if isinstance(golden_text, str) and golden_text.startswith('['):
        try:
            import ast
            golden_list = ast.literal_eval(golden_text)
        except (SyntaxError, ValueError):
            golden_list = [golden_text]
    else:
        golden_list = [golden_text]

    golden_embeds = np.array(embedding.embed_documents(golden_list))
    chunk_texts = [doc.page_content for doc, _ in retrieved_docs]

    if not chunk_texts:
        return 0, 0.0, "No documents retrieved"

    chunk_embeds = np.array(embedding.embed_documents(chunk_texts))
    sim_matrix = cosine_similarity(golden_embeds, chunk_embeds)

    best_chunk_idx = sim_matrix.max(axis=0).argmax()
    best_golden_idx = sim_matrix.max(axis=1).argmax()
    best_sim = sim_matrix.max()

    if best_sim >= threshold:
        best_rank = int(np.where(sim_matrix[best_golden_idx] == sim_matrix[best_golden_idx].max())[0][0]) + 1
        return 1, best_sim, f"Matched at Rank {best_rank} (similarity: {best_sim:.4f})"
    else:
        return 0, best_sim, f"Best similarity was {best_sim:.4f} (below {threshold} threshold)"

# ==========================================
# 3. RUN THE THRESHOLD BAKE-OFF
# ==========================================
thresholds_to_test = [0.6, 0.7, 0.75, 0.8, 0.81, 0.82]
results_data = []

for score_threshold in thresholds_to_test:
    print(f"\n{'='*80}")
    print(f"Testing Score Threshold: {score_threshold}")
    print(f"{'='*80}")

    prof_hits, lay_hits = 0, 0

    for _, row in test_queries.iterrows():
        persona = row['Persona']
        query = row['Question']
        golden_ref = row['Reference']

        try:
            docs_and_scores = vectorstore.similarity_search_with_relevance_scores(
                query, k=20, score_threshold=score_threshold
            )
            num_docs = len(docs_and_scores)

            is_hit, best_sim, match_reason = check_hit_fuzzy(golden_ref, docs_and_scores)

            if persona == 'Professional':
                prof_hits += is_hit
            else:
                lay_hits += is_hit

            status = "HIT " if is_hit else "MISS"
            print(f"[{persona[:4].upper()}] {status} | Docs: {num_docs} | Sim: {best_sim:.4f} | Q: {query[:50]}...")
            print(f"       Result: {match_reason}\n")

        except Exception as e:
            print(f"[{persona[:4].upper()}] ERR  | Docs: 0 | Q: {query[:50]}...")
            print(f"       Result: 0 documents met the {score_threshold} threshold.\n")

    results_data.append({
        "Threshold": score_threshold,
        "Prof Hit Rate": prof_hits / 5,
        "Layperson Hit Rate": lay_hits / 5,
    })

# ==========================================
# 4. DISPLAY SUMMARY
# ==========================================
print("\nTHRESHOLD PERFORMANCE SUMMARY")
display(pd.DataFrame(results_data))


Testing Score Threshold: 0.6
[PROF] HIT  | Docs: 20 | Sim: 0.8889 | Q: What are the primary tools used to assess glycemic...
       Result: Matched at Rank 1 (similarity: 0.8889)

[PROF] HIT  | Docs: 20 | Sim: 0.8913 | Q: When might a less stringent A1C goal be appropriat...
       Result: Matched at Rank 2 (similarity: 0.8913)

[PROF] HIT  | Docs: 20 | Sim: 0.8192 | Q: Mr. Tan, 52 years old, presents to your clinic aft...
       Result: Matched at Rank 13 (similarity: 0.8192)

[PROF] HIT  | Docs: 20 | Sim: 0.9112 | Q: A patient with type 2 diabetes on an SGLT2 inhibit...
       Result: Matched at Rank 5 (similarity: 0.9112)

[PROF] HIT  | Docs: 20 | Sim: 0.9591 | Q: A newly diagnosed type 2 diabetes patient has CKD ...
       Result: Matched at Rank 2 (similarity: 0.9591)

[LAYP] HIT  | Docs: 20 | Sim: 0.8735 | Q: "My HbA1c has improved but my sensor shows I am ha...
       Result: Matched at Rank 1 (similarity: 0.8735)

[LAYP] HIT  | Docs: 2 | Sim: 0.7990 | Q: "My doctor wants me to

[PROF] HIT  | Docs: 20 | Sim: 0.8913 | Q: When might a less stringent A1C goal be appropriat...
       Result: Matched at Rank 2 (similarity: 0.8913)

[PROF] MISS | Docs: 0 | Sim: 0.0000 | Q: Mr. Tan, 52 years old, presents to your clinic aft...
       Result: No documents retrieved

[PROF] HIT  | Docs: 12 | Sim: 0.9112 | Q: A patient with type 2 diabetes on an SGLT2 inhibit...
       Result: Matched at Rank 5 (similarity: 0.9112)



[PROF] HIT  | Docs: 20 | Sim: 0.9591 | Q: A newly diagnosed type 2 diabetes patient has CKD ...
       Result: Matched at Rank 2 (similarity: 0.9591)

[LAYP] MISS | Docs: 0 | Sim: 0.0000 | Q: "My HbA1c has improved but my sensor shows I am ha...
       Result: No documents retrieved

[LAYP] MISS | Docs: 0 | Sim: 0.0000 | Q: "My doctor wants me to wear a glucose sensor on my...
       Result: No documents retrieved

[LAYP] MISS | Docs: 0 | Sim: 0.0000 | Q: "I am going in for surgery next week. Do I need to...
       Result: No documents retrieved



[LAYP] MISS | Docs: 0 | Sim: 0.0000 | Q: "My father is very ill with cancer and diabetes. H...
       Result: No documents retrieved

[LAYP] MISS | Docs: 0 | Sim: 0.0000 | Q: "I cannot afford the glucose sensor. What can I do...
       Result: No documents retrieved


Testing Score Threshold: 0.75
[PROF] MISS | Docs: 0 | Sim: 0.0000 | Q: What are the primary tools used to assess glycemic...
       Result: No documents retrieved

[PROF] HIT  | Docs: 9 | Sim: 0.8913 | Q: When might a less stringent A1C goal be appropriat...
       Result: Matched at Rank 2 (similarity: 0.8913)



[PROF] MISS | Docs: 0 | Sim: 0.0000 | Q: Mr. Tan, 52 years old, presents to your clinic aft...
       Result: No documents retrieved

[PROF] HIT  | Docs: 2 | Sim: 0.8279 | Q: A patient with type 2 diabetes on an SGLT2 inhibit...
       Result: Matched at Rank 1 (similarity: 0.8279)



...

[PROF] MISS | Docs: 0 | Sim: 0.0000 | Q: What are the primary tools used to assess glycemic...
       Result: No documents retrieved

[PROF] HIT  | Docs: 1 | Sim: 0.8430 | Q: When might a less stringent A1C goal be appropriat...
       Result: Matched at Rank 1 (similarity: 0.8430)



[PROF] MISS | Docs: 0 | Sim: 0.0000 | Q: Mr. Tan, 52 years old, presents to your clinic aft...
       Result: No documents retrieved

[PROF] MISS | Docs: 0 | Sim: 0.0000 | Q: A patient with type 2 diabetes on an SGLT2 inhibit...
       Result: No documents retrieved

[PROF] HIT  | Docs: 1 | Sim: 0.9114 | Q: A newly diagnosed type 2 diabetes patient has CKD ...
       Result: Matched at Rank 1 (similarity: 0.9114)



[LAYP] MISS | Docs: 0 | Sim: 0.0000 | Q: "My HbA1c has improved but my sensor shows I am ha...
       Result: No documents retrieved

[LAYP] MISS | Docs: 0 | Sim: 0.0000 | Q: "My doctor wants me to wear a glucose sensor on my...
       Result: No documents retrieved

[LAYP] MISS | Docs: 0 | Sim: 0.0000 | Q: "I am going in for surgery next week. Do I need to...
       Result: No documents retrieved



[LAYP] MISS | Docs: 0 | Sim: 0.0000 | Q: "My father is very ill with cancer and diabetes. H...
       Result: No documents retrieved

[LAYP] MISS | Docs: 0 | Sim: 0.0000 | Q: "I cannot afford the glucose sensor. What can I do...
       Result: No documents retrieved


THRESHOLD PERFORMANCE SUMMARY


,Threshold,Prof Hit Rate,Layperson Hit Rate
0,0.60,1.0,0.8
1,0.70,0.8,0.0
2,0.75,0.6,0.0
3,0.80,0.4,0.0
4,0.81,0.4,0.0
5,0.82,0.4,0.0


#### Layperson queries hit a cliff at 0.70. They go from 80% hit rate to 0%. The only viable threshold for both personas is 0.60.

### We will also make use of the relevance score to figure out a suitable threshold.

In [ ]:
import pandas as pd

thresholds_to_test = [0.6, 0.65, 0.7, 0.75, 0.8, 0.81, 0.82, 0.83, 0.84, 0.85]

prof_sample = retrieved_data_df[retrieved_data_df['Persona'] == 'Professional'].sample(n=5, random_state=42)
layperson_sample = retrieved_data_df[retrieved_data_df['Persona'] == 'Layperson'].sample(n=5, random_state=42)
test_queries = pd.concat([prof_sample, layperson_sample])

results = []

for threshold in thresholds_to_test:
    for _, row in test_queries.iterrows():
        query = row['Question']
        persona = row['Persona']

        docs_and_scores = vectorstore.similarity_search_with_relevance_scores(
            query, k=5, score_threshold=threshold
        )

        num_retrieved = len(docs_and_scores)
        scores = [score for _, score in docs_and_scores]

        results.append({
            'threshold': threshold,
            'persona': persona,
            'question': query[:80],
            'num_retrieved': num_retrieved,
            'avg_score': sum(scores) / len(scores) if scores else 0,
            'min_score': min(scores) if scores else 0,
            'max_score': max(scores) if scores else 0,
        })

results_df = pd.DataFrame(results)

# Summary: average docs retrieved and average score per threshold and persona
summary = results_df.groupby(['threshold', 'persona']).agg(
    avg_docs_retrieved=('num_retrieved', 'mean'),
    avg_score=('avg_score', 'mean'),
    zero_result_queries=('num_retrieved', lambda x: (x == 0).sum()),
).reset_index()

print("=== Threshold Comparison ===\n")
display(summary)

# Overall summary per threshold (both personas combined)
overall = results_df.groupby('threshold').agg(
    avg_docs=('num_retrieved', 'mean'),
    zero_results=('num_retrieved', lambda x: (x == 0).sum()),
    avg_score=('avg_score', 'mean'),
).reset_index()

print("\n=== Overall (best threshold = high avg_docs + zero zero_results + high avg_score) ===\n")
display(overall)

# Pick the best: highest threshold that still returns docs for all 10 queries
best_candidates = overall[overall['zero_results'] == 0]
if not best_candidates.empty:
    best = best_candidates['threshold'].max()
    print(f"\nRecommended threshold: {best}")
    print("(Highest threshold with zero failed queries across both personas)")
else:
    best = overall.loc[overall['zero_results'].idxmin(), 'threshold']
    print(f"\nRecommended threshold: {best}")
    print("(No threshold returned results for all queries; picked the one with fewest failures)")

=== Threshold Comparison ===



,threshold,persona,avg_docs_retrieved,avg_score,zero_result_queries
0,0.60,Layperson,3.6,0.645401,0
1,0.60,Professional,5.0,0.749309,0
2,0.65,Layperson,2.0,0.269759,3
3,0.65,Professional,5.0,0.749309,0
4,0.70,Layperson,0.0,0.000000,5
5,0.70,Professional,4.0,0.616256,1
6,0.75,Layperson,0.0,0.000000,5
7,0.75,Professional,2.4,0.477474,2
8,0.80,Layperson,0.0,0.000000,5
9,0.80,Professional,1.0,0.329642,3



=== Overall (best threshold = high avg_docs + zero zero_results + high avg_score) ===



,threshold,avg_docs,zero_results,avg_score
0,0.60,4.3,0,0.697355
1,0.65,3.5,3,0.509534
2,0.70,2.0,6,0.308128
3,0.75,1.2,7,0.238737
4,0.80,0.5,8,0.164821
5,0.81,0.3,8,0.167445
6,0.82,0.2,8,0.169163
7,0.83,0.2,8,0.169163
8,0.84,0.2,8,0.169163
9,0.85,0.0,10,0.000000



Recommended threshold: 0.6
(Highest threshold with zero failed queries across both personas)


In [ ]:
import pandas as pd
import numpy as np

# ==========================================
# 1. SAMPLE THE GOLDEN DATASET
# ==========================================
print("⏳ Sampling Data...")
# Assuming your dataframe is named retrieved_data_df
prof_sample = retrieved_data_df[retrieved_data_df['Persona'] == 'Professional'].sample(n=5, random_state=42)
layperson_sample = retrieved_data_df[retrieved_data_df['Persona'] == 'Layperson'].sample(n=5, random_state=42)
test_queries = pd.concat([prof_sample, layperson_sample])

# ==========================================
# 2. RUN SCORE PROFILING (No Threshold)
# ==========================================
print("⏳ Profiling natural relevance scores...")
results_data = []

for idx, row in test_queries.iterrows():
    persona = row['Persona']
    query = row['Question']

    try:
        # We query for top 5 without a threshold to see the natural score distribution
        docs_and_scores = vectorstore.similarity_search_with_relevance_scores(
            query,
            k=5
        )

        # Extract just the mathematical scores
        scores = [score for doc, score in docs_and_scores]

        if scores:
            results_data.append({
                "Persona": persona,
                "Query": query[:60] + "...",
                "Top_1_Score": scores[0],           # The best matching document
                "Top_5_Avg_Score": np.mean(scores), # The average of the top 5
                "Rank_5_Score": scores[-1]          # The worst matching document in the top 5
            })

    except Exception as e:
        print(f"Error processing query: {e}")

# ==========================================
# 3. DISPLAY RESULTS & CALCULATE OPTIMAL THRESHOLD
# ==========================================
results_df = pd.DataFrame(results_data)

print("\n📊 SCORE DISTRIBUTION PER QUERY:")
display(results_df)

# Calculate the minimum top-1 score for each persona
prof_min_top1 = results_df[results_df['Persona'] == 'Professional']['Top_1_Score'].min()
lay_min_top1 = results_df[results_df['Persona'] == 'Layperson']['Top_1_Score'].min()

print("\n" + "="*80)
print("🎯 THRESHOLD RECOMMENDATION ENGINE")
print("="*80)
print(f"Lowest Top-1 Score (Professional) : {prof_min_top1:.4f}")
print(f"Lowest Top-1 Score (Layperson)    : {lay_min_top1:.4f}")
print("-" * 80)

# The optimal threshold is usually slightly below the lowest Top-1 score
# to ensure every query returns at least 1 document.
safe_threshold = min(prof_min_top1, lay_min_top1) - 0.02

print(f"💡 RECOMMENDED GLOBAL THRESHOLD: {safe_threshold:.2f}")
print("\nWhy?")
print(f"If you set your threshold any higher than {min(prof_min_top1, lay_min_top1):.4f}, some Layperson ")
print("queries will return exactly ZERO documents. Our Golden Eval Set is meticulously engineered so that we should be able to retrieve relevant chunks.")

⏳ Sampling Data...
⏳ Profiling natural relevance scores...

📊 SCORE DISTRIBUTION PER QUERY:


,Persona,Query,Top_1_Score,Top_5_Avg_Score,Rank_5_Score
0,Professional,What are the primary tools used to assess glyc...,0.746183,0.723864,0.711958
1,Professional,When might a less stringent A1C goal be approp...,0.844305,0.793224,0.762637
2,Professional,"Mr. Tan, 52 years old, presents to your clinic...",0.687149,0.665263,0.653186
3,Professional,A patient with type 2 diabetes on an SGLT2 inh...,0.793537,0.757520,0.731614
4,Professional,A newly diagnosed type 2 diabetes patient has ...,0.847329,0.806674,0.781884
5,Layperson,"""My HbA1c has improved but my sensor shows I a...",0.687369,0.674793,0.658026
6,Layperson,"""My doctor wants me to wear a glucose sensor o...",0.608043,0.597782,0.587787
7,Layperson,"""I am going in for surgery next week. Do I nee...",0.689092,0.674000,0.659812
8,Layperson,"""My father is very ill with cancer and diabete...",0.649887,0.641142,0.634728
9,Layperson,"""I cannot afford the glucose sensor. What can ...",0.631655,0.591894,0.575401



🎯 THRESHOLD RECOMMENDATION ENGINE
Lowest Top-1 Score (Professional) : 0.6871
Lowest Top-1 Score (Layperson)    : 0.6080
--------------------------------------------------------------------------------
💡 RECOMMENDED GLOBAL THRESHOLD: 0.59

Why?
If you set your threshold any higher than 0.6080, some Layperson 
queries will return exactly ZERO documents. Our Golden Eval Set is meticulously engineered so that we should be able to retrieve relevant chunks.


### Threshold = 0.6 seems to be the sweet spot given the current db and for conversational queries. For strictly technical queries, the similarity scores were even higher so we only have to worry about conversational queries. We will use 0.6 as our safety net, so at least the conversational queries will at least have one document retrieved. Now let's compare other two search methods with similarity threshold = 0.6

In [ ]:
import pandas as pd
import ast
import numpy as np
from datasets import Dataset
from ragas import evaluate
from ragas.metrics import context_precision

# ==========================================
# 1. SETUP HELPERS & RAGAS FUNCTION
# ==========================================
def safe_parse(val, force_list=False):
    if isinstance(val, list):
        return val
    try:
        parsed = ast.literal_eval(val)
        if isinstance(parsed, list):
            return parsed
    except:
        pass
    return [str(val)] if force_list else val

def run_retrieval(df: pd.DataFrame) -> pd.DataFrame:
    # Prepare the dataframe for Ragas
    df_eval = df.copy()
    df_eval['Retrieved Contexts'] = df_eval['Retrieved Contexts'].apply(lambda x: safe_parse(x, force_list=True))
    df_eval['Reference'] = df_eval['Reference'].apply(lambda x: safe_parse(x, force_list=True))

    df_eval['joined_reference'] = df_eval['Reference'].apply(
        lambda x: "\n".join(x) if isinstance(x, list) else str(x)
    )

    hf_dataset = Dataset.from_pandas(df_eval.rename(columns={
        'Question': 'user_input',
        'Retrieved Contexts': 'retrieved_contexts',
        'joined_reference': 'reference'
    }))

    # Run the LLM Judge
    retrieval_results = evaluate(
        dataset=hf_dataset,
        metrics=[context_precision],
        llm=evaluator_llm,
        raise_exceptions=False # Prevents crashing if a specific query fails
    )

    retrieval_df = retrieval_results.to_pandas()
    return retrieval_df['context_precision']

# ==========================================
# 2. SAMPLE 10 QUERIES
# ==========================================
print("⏳ Sampling 10 Queries from Golden Dataset...")
prof_sample = retrieved_data_df[retrieved_data_df['Persona'] == 'Professional'].sample(n=5, random_state=42)
layperson_sample = retrieved_data_df[retrieved_data_df['Persona'] == 'Layperson'].sample(n=5, random_state=42)
test_queries = pd.concat([prof_sample, layperson_sample]).copy()

# ==========================================
# 3. INITIALIZE THE 3 RETRIEVERS
# ==========================================
# Make sure vectorstore is correctly pointing to 'ADA_Diabetes_db_20260312'
LOCKED_THRESHOLD = 0.6

retrievers = {
    "Standard Similarity": vectorstore.as_retriever(
        search_type="similarity",
        search_kwargs={"k": 5}
    ),
    f"Score Threshold ({LOCKED_THRESHOLD})": vectorstore.as_retriever(
        search_type="similarity_score_threshold",
        search_kwargs={"score_threshold": LOCKED_THRESHOLD, "k": 5}
    ),
    "MMR (Diversity)": vectorstore.as_retriever(
        search_type="mmr",
        search_kwargs={"k": 5, "fetch_k": 20, "lambda_mult": 0.5}
    )
}

# ==========================================
# 4. RUN THE BAKE-OFF BATCH
# ==========================================
all_results = test_queries[['Persona', 'Question']].copy()

for name, retriever in retrievers.items():
    print(f"\n{'='*60}")
    print(f"🚀 Evaluating Strategy: {name}")
    print(f"{'='*60}")

    retrieved_contexts_list = []

    # Fetch documents for all 10 queries using this specific retriever
    for idx, row in test_queries.iterrows():
        try:
            docs = retriever.invoke(row['Question'])
            # Extract raw text, handling empty retrievals (if threshold drops all docs)
            chunk_texts = [doc.page_content for doc in docs] if docs else [""]
            retrieved_contexts_list.append(chunk_texts)
        except Exception as e:
            # Fallback for thresholding errors
            retrieved_contexts_list.append([""])

    # Temporarily add contexts to the dataframe to pass to RAGAS
    temp_df = test_queries.copy()
    temp_df['Retrieved Contexts'] = retrieved_contexts_list

    # Run RAGAS and save the scores back to our main leaderboard dataframe
    print(f"🧠 Passing {name} retrieved chunks to LLM Judge...")
    scores = run_retrieval(temp_df)

    # Save the scores under a column named after the strategy
    all_results[f"{name} Score"] = scores.values

# ==========================================
# 5. DISPLAY LEADERBOARD
# ==========================================
print("\n\n🏆 FINAL RAGAS LEADERBOARD (Context Precision)")
display(all_results)

# Calculate Average Scores grouped by Persona to see which strategy wins for which user!
summary_df = all_results.groupby('Persona')[[
    "Standard Similarity Score",
    f"Score Threshold ({LOCKED_THRESHOLD}) Score",
    "MMR (Diversity) Score"
]].mean()

print("\n📊 AVERAGE SCORES BY PERSONA")
display(summary_df)

/tmp/ipykernel_14945/3613857057.py:6: DeprecationWarning: Importing context_precision from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import context_precision
  from ragas.metrics import context_precision


⏳ Sampling 10 Queries from Golden Dataset...

🚀 Evaluating Strategy: Standard Similarity
🧠 Passing Standard Similarity retrieved chunks to LLM Judge...


Evaluating:   0%|          | 0/10 [00:00<?, ?it/s]


🚀 Evaluating Strategy: Score Threshold (0.6)
🧠 Passing Score Threshold (0.6) retrieved chunks to LLM Judge...


Evaluating:   0%|          | 0/10 [00:00<?, ?it/s]

...

Evaluating:   0%|          | 0/10 [00:00<?, ?it/s]



🏆 FINAL RAGAS LEADERBOARD (Context Precision)


,Persona,Question,Standard Similarity Score,Score Threshold (0.6) Score,MMR (Diversity) Score
0,Professional,What are the primary tools used to assess glyc...,0.755556,0.804167,1.000000
5,Professional,When might a less stringent A1C goal be approp...,0.887500,0.887500,0.679167
82,Professional,"Mr. Tan, 52 years old, presents to your clinic...",1.000000,1.000000,0.887500
31,Professional,A patient with type 2 diabetes on an SGLT2 inh...,1.000000,1.000000,1.000000
13,Professional,A newly diagnosed type 2 diabetes patient has ...,0.950000,0.950000,0.805556
69,Layperson,"""My HbA1c has improved but my sensor shows I a...",0.804167,0.804167,0.916667
47,Layperson,"""My doctor wants me to wear a glucose sensor o...",1.000000,1.000000,0.950000
60,Layperson,"""I am going in for surgery next week. Do I nee...",0.950000,0.950000,1.000000
64,Layperson,"""My father is very ill with cancer and diabete...",1.000000,1.000000,1.000000
50,Layperson,"""I cannot afford the glucose sensor. What can ...",0.500000,0.000000,0.500000



📊 AVERAGE SCORES BY PERSONA


,Standard Similarity Score,Score Threshold (0.6) Score,MMR (Diversity) Score
Persona,,,
Layperson,0.850833,0.750833,0.873333
Professional,0.918611,0.928333,0.874444


## Conclusion
Similarity Thresholding = 0.6 is the best search type for Professional queries while MMR turns out to perform best against Layperson queries.
- Technical terms match VDB well; threshold helps filtering noises
- Informal language needs soft matching; threshold causes zero-result failures

# Experimenting BM25 Keyword Retriever

BM25 excels at matching **exact medical terms** (e.g., "IBS-C", "HbA1c", "metformin") that dense embeddings may dilute. We build a BM25 index over all documents in the ChromaDB collection and wrap it as a LangChain retriever.

### We will use BM25 Relevance Score to evaluate the performances.

In [ ]:
# Extract all documents from ChromaDB for BM25 indexing
all_data = collection.get(include=['documents', 'metadatas'])

all_doc_texts = all_data['documents']
all_doc_metadatas = all_data['metadatas']
all_doc_ids = all_data['ids']

print(f"Total documents for BM25 indexing: {len(all_doc_texts)}")
print(f"Sample document preview: {all_doc_texts[10][:20000]}...")

Total documents for BM25 indexing: 3022
Sample document preview: oses more people with prediabetes and diabetes (4). Moreover, the efficacy of interventions for primary prevention of type 2 diabetes (i.e., preventing conversion of prediabetes to type 2 diabetes) has been demonstrated mainly among individuals with prediabetes who have impaired glucose tolerance (IGT) with or without elevated fasting glucose, not for individuals with isolated impaired fasting glucose (IFG) or for those with prediabetes defined by A1C criteria (5–8). The same tests may be used t...


## Testing Different Index Strategy

### We will use .get_scores() from the rank_bm25 library to measure the relevance score.

### Naive indexing includes "junk" words (like the, is, and, i). These words appear everywhere in medical texts, so they add points to almost every document, making scores artificially high. We will try to improve our tokenizer to filter out the noise terms like "what", "are", "the", etc.

In [ ]:
import re
import numpy as np

# --- 1. Define the two tokenization strategies ---

def naive_tokenize(text):
    return text.lower().split()

STOP_WORDS = {"what", "are", "the", "of", "a", "an", "is", "was", "were",
              "and", "or", "but", "in", "on", "at", "to", "for", "with",
              "it", "its", "kind", "can", "i", "ive", "been", "having",
              "too", "this", "do", "my", "so", "get"}

def refined_tokenize(text):
    clean_text = re.sub(r'[^\w\s]', '', text.lower())
    return [word for word in clean_text.split() if word not in STOP_WORDS]

# --- 2. Build both indexes ---
print("Building Naive Index...")
corpus_naive = [naive_tokenize(doc) for doc in all_doc_texts]
bm25_naive = BM25Okapi(corpus_naive, k1=1.5, b=0.75)

print("Building Refined Index...")
corpus_refined = [refined_tokenize(doc) for doc in all_doc_texts]
bm25_refined = BM25Okapi(corpus_refined, k1=1.5, b=0.75)

# --- 3. Sample queries from the golden eval set ---
prof_sample = retrieved_data_df[retrieved_data_df['Persona'] == 'Professional'].sample(n=5, random_state=42)
layperson_sample = retrieved_data_df[retrieved_data_df['Persona'] == 'Layperson'].sample(n=5, random_state=42)
test_batch = pd.concat([prof_sample, layperson_sample])

# --- 4. Run the Bake-Off ---
def print_top_3(scores, name, tokens):
    print(f"\n  Strategy: {name}")
    print(f"  Tokens Searched: {tokens}")
    top_indices = np.argsort(scores)[::-1][:3]
    for rank, idx in enumerate(top_indices):
        if scores[idx] > 0:
            snippet = all_doc_texts[idx].replace('\n', ' ')[:250]
            print(f"    Rank {rank+1} | Score: {scores[idx]:.2f} | {snippet}...")

for _, row in test_batch.iterrows():
    persona = row['Persona']
    query_text = row['Question']

    print(f"\n{'='*90}")
    print(f"TESTING [{persona.upper()}] QUERY")
    print(f"Question: \"{query_text[:120]}...\"")
    print(f"{'='*90}")

    query_naive = naive_tokenize(query_text)
    query_refined = refined_tokenize(query_text)

    scores_naive = bm25_naive.get_scores(query_naive)
    scores_refined = bm25_refined.get_scores(query_refined)

    print_top_3(scores_naive, "Naive Indexing", query_naive)
    print_top_3(scores_refined, "Refined Indexing", query_refined)

Building Naive Index...
Building Refined Index...

TESTING [PROFESSIONAL] QUERY
Question: "What are the primary tools used to assess glycemic status in a patient with diabetes?..."

  Strategy: Naive Indexing
  Tokens Searched: ['what', 'are', 'the', 'primary', 'tools', 'used', 'to', 'assess', 'glycemic', 'status', 'in', 'a', 'patient', 'with', 'diabetes?']
    Rank 1 | Score: 24.30 |  growing, especially in people who are taking insulin. TIR is a useful metric of glycemic status. A 10- to 14-day CGM assessment of TIR, with CGM wear of 70% or higher, and other CGM metrics can be used to assess glycemic status and are useful in cli...
    Rank 2 | Score: 21.81 | and follow-up, as appropriate, to: • Confirm the diagnosis and classify diabetes. A • Assess glycemic status and previous and current treatment. A • Evaluate for diabetes complications, potential comorbid conditions, and overall health status. A • Id...
    Rank 3 | Score: 21.49 | diabetes management (rtCGM) and over-the-counter

### The refined tokenizer still needs improving. It sometimes handles specific terms poorly, such as turning "trimester-specific" into 'trimesterspecific'.

In [ ]:
import re

STOP_WORDS = {"what", "are", "the", "of", "a", "an", "is", "was", "were",
              "and", "or", "but", "in", "on", "at", "to", "for", "with",
              "it", "its", "kind", "can", "i", "ive", "been", "having",
              "too", "this", "do", "my", "so", "get", "about", "how", "why"}

def smart_tokenize(text):
    # 1. Lowercase for case-insensitivity
    text = text.lower()

    # 2. Convert hyphens, slashes, and colons into SPACES first
    # "fda-approved" -> "fda approved"
    # "15197:2013" -> "15197 2013"
    text = re.sub(r'[-/:]', ' ', text)

    # 3. Split the text into raw tokens based on the new spaces
    raw_tokens = text.split()

    smart_tokens = []
    for token in raw_tokens:
        # 4. Strip ONLY outside punctuation.
        # This safely removes commas at the end of sentences,
        # but keeps internal periods safe (e.g., "u.s." becomes "u.s")
        clean_token = token.strip('.,;!?()"\'[]{}')

        # 5. Keep the token if it has length and isn't a stop word
        if clean_token and clean_token not in STOP_WORDS:
            smart_tokens.append(clean_token)

    return smart_tokens

# --- Quick Test to prove it works ---
test_string = "The U.S. FDA-approved meter (ISO 15197:2013) is used for anti-PD-1/anti-PD-L1 patients."
print(f"Old Refined output : {refined_tokenize(test_string)}")
print(f"Smart Token output : {smart_tokenize(test_string)}")

Old Refined output : ['us', 'fdaapproved', 'meter', 'iso', '151972013', 'used', 'antipd1antipdl1', 'patients']
Smart Token output : ['u.s', 'fda', 'approved', 'meter', 'iso', '15197', '2013', 'used', 'anti', 'pd', '1', 'anti', 'pd', 'l1', 'patients']


### Further refining the tokenizer so 'U.S.' is tokenized as 'u.s.', and 'FDA-approved' is tokenized as 'fda', 'approved'.

In [ ]:
import re

STOP_WORDS = {"what", "are", "the", "of", "a", "an", "is", "was", "were",
              "and", "or", "but", "in", "on", "at", "to", "for", "with",
              "it", "its", "kind", "can", "i", "ive", "been", "having",
              "too", "this", "do", "my", "so", "get", "about", "how", "why"}

def ultimate_tokenize(text):
    text = text.lower()

    # 1. Replace hyphens, slashes, and colons with spaces
    text = re.sub(r'[-/:]', ' ', text)

    raw_tokens = text.split()
    final_tokens = []

    for token in raw_tokens:
        # 2. Strip all punctuation EXCEPT the period
        clean_token = token.strip(',;!?()"\'[]{}')

        # 3. Smart Period Handling:
        # If the word ends with a period, but only has ONE period total,
        # it is just the end of a sentence (e.g., "patients."). Chop it off.
        if clean_token.endswith('.') and clean_token.count('.') == 1:
            clean_token = clean_token[:-1]

        # (If it has multiple periods like "u.s.", the code above ignores it and keeps the dots!)

        # 4. Filter out stop words and empty strings
        if clean_token and clean_token not in STOP_WORDS:
            final_tokens.append(clean_token)

    return final_tokens

# --- Quick Test ---
test_string = "The U.S. FDA-approved meter (ISO 15197:2013) is used for anti-PD-1/anti-PD-L1 patients."
print(f"Ultimate Token output : {ultimate_tokenize(test_string)}")

Ultimate Token output : ['u.s.', 'fda', 'approved', 'meter', 'iso', '15197', '2013', 'used', 'anti', 'pd', '1', 'anti', 'pd', 'l1', 'patients']


In [ ]:
import random
import re
import numpy as np
from rank_bm25 import BM25Okapi

random.seed(42)

# ==========================================
# 1. DEFINE TOKENIZERS
# ==========================================

STOP_WORDS = {"what", "are", "the", "of", "a", "an", "is", "was", "were",
              "and", "or", "but", "in", "on", "at", "to", "for", "with",
              "it", "its", "kind", "can", "i", "ive", "been", "having",
              "too", "this", "do", "my", "so", "get", "about", "how", "why"}

def naive_tokenize(text):
    return text.lower().split()

def ultimate_tokenize(text):
    text = text.lower()
    text = re.sub(r'[-/:]', ' ', text)
    raw_tokens = text.split()
    final_tokens = []

    for token in raw_tokens:
        clean_token = token.strip(',;!?()\"\'[]{}')

        if clean_token.endswith('.') and clean_token.count('.') == 1:
            clean_token = clean_token[:-1]

        if clean_token and clean_token not in STOP_WORDS:
            final_tokens.append(clean_token)

    return final_tokens

# ==========================================
# 2. BUILD THE REMAINING INDEXES
# ==========================================
print("Building Naive Index (Control)...")
corpus_naive = [naive_tokenize(doc) for doc in all_doc_texts]
bm25_naive = BM25Okapi(corpus_naive, k1=1.5, b=0.75)

print("Building Ultimate Index (Optimized)...")
corpus_ultimate = [ultimate_tokenize(doc) for doc in all_doc_texts]
bm25_ultimate = BM25Okapi(corpus_ultimate, k1=1.5, b=0.75)

# ==========================================
# 3. SAMPLE FROM GOLDEN EVAL SET
# ==========================================
prof_sample = retrieved_data_df[retrieved_data_df['Persona'] == 'Professional'].sample(n=5, random_state=42)
layperson_sample = retrieved_data_df[retrieved_data_df['Persona'] == 'Layperson'].sample(n=5, random_state=42)
test_batch = pd.concat([prof_sample, layperson_sample])

# ==========================================
# 4. CLEAN BAKE-OFF LOOP
# ==========================================
def print_top_3(scores, name, tokens):
    print(f"\n  Strategy: {name}")
    print(f"  Tokens Searched: {tokens}")
    top_indices = np.argsort(scores)[::-1][:3]
    for rank, idx in enumerate(top_indices):
        if scores[idx] > 0:
            snippet = all_doc_texts[idx].replace('\n', ' ')[:250]
            print(f"    Rank {rank+1} | Score: {scores[idx]:.2f} | {snippet}...")

for _, row in test_batch.iterrows():
    persona = row['Persona']
    query_text = row['Question']

    print(f"\n{'='*90}")
    print(f"TESTING [{persona.upper()}] QUERY")
    print(f"Question: \"{query_text[:120]}...\"")
    print(f"{'='*90}")

    q_naive = naive_tokenize(query_text)
    q_ultimate = ultimate_tokenize(query_text)

    scores_naive = bm25_naive.get_scores(q_naive)
    scores_ultimate = bm25_ultimate.get_scores(q_ultimate)

    print_top_3(scores_naive, "Naive Indexing", q_naive)
    print_top_3(scores_ultimate, "Ultimate Indexing", q_ultimate)

Building Naive Index (Control)...
Building Ultimate Index (Optimized)...

TESTING [PROFESSIONAL] QUERY
Question: "What are the primary tools used to assess glycemic status in a patient with diabetes?..."

  Strategy: Naive Indexing
  Tokens Searched: ['what', 'are', 'the', 'primary', 'tools', 'used', 'to', 'assess', 'glycemic', 'status', 'in', 'a', 'patient', 'with', 'diabetes?']
    Rank 1 | Score: 24.30 |  growing, especially in people who are taking insulin. TIR is a useful metric of glycemic status. A 10- to 14-day CGM assessment of TIR, with CGM wear of 70% or higher, and other CGM metrics can be used to assess glycemic status and are useful in cli...
    Rank 2 | Score: 21.81 | and follow-up, as appropriate, to: • Confirm the diagnosis and classify diabetes. A • Assess glycemic status and previous and current treatment. A • Evaluate for diabetes complications, potential comorbid conditions, and overall health status. A • Id...
    Rank 3 | Score: 21.49 | diabetes management (rtCG

#### We eyeball the results and manually assess if the fetched docs are actually better or not. Below are our results.

#### Professional Queries (5/5)

|Query|Naive Top Result|Ultimate Top Result|Better?|
|---|---|---|---|
|Glycemic status tools|TIR/CGM metric (decent)|"Assess glycemic status and treatment" (direct match)|Ultimate|
|Less stringent A1C|DCCT/UKPDS + individualized goals|Same document|Tie|
|Mr. Tan vignette|Cardiovascular evolocumab trial (irrelevant)|Hypoglycemia + eGFR/creatinine (closer to patient's labs)|Ultimate|
|SGLT2 before surgery|Medication simplification in elderly (wrong topic)|Rank 3 literally says "SGLT2 inhibitors should be held for 3-4 days before elective surgery"|Ultimate|
|CKD eGFR 40|SGLT2 for CKD (relevant)|SGLT2 with eGFR as low as 20 mL/min (relevant, higher score)|Ultimate|

#### Layperson Queries (5/5)

|Query|Naive Top Result|Ultimate Top Result|Better?|
|---|---|---|---|
|HbA1c vs sensor at night|FIB-4 fibrosis score (irrelevant)|Same irrelevant result|Tie (both poor)|
|Wear glucose sensor?|Overweight/obesity appointments (irrelevant)|CGM accuracy on day 1 of sensor wear (on-topic)|Ultimate|
|Stop meds before surgery?|Prediabetes progression (irrelevant)|Behavioral self-care difficulties (irrelevant)|Tie (both poor)|
|Father ill, tight control?|Type 1 lifestyle factors (irrelevant)|NICE-SUGAR trial: aggressive vs moderate glycemic goals for critically ill (perfect match)|Ultimate|
|Can't afford sensor|Monogenic diabetes model (irrelevant)|Sensor substance interactions (irrelevant)|Tie (both poor)|


#### Why the scores are lower but the results are better

Naive indexing inflates scores with stop words. Look at the Mr. Tan vignette:

- Naive: 116 tokens searched, including `"a"`, `"the"`, `"to"`, `"is"`, `"with"`, `"in"`, `"for"`, `"was"` (each appearing multiple times). Score: 68.89 -- but the top result is about evolocumab cardiovascular trials, which has nothing to do with a SOAP assessment. The score is high because common words like "he", "was", "with", "type", "2" match almost every document.
- Ultimate: 105 tokens searched, stop words removed. Score: 54.03 -- lower, but the results are about hypoglycemia management and eGFR/creatinine levels, which are actually relevant to the patient's clinical picture.

Stop words act as score inflators that reward long, generic documents rather than topically relevant ones.

#### Conclusion
We will stick with the refined indexing with stop words excluded.

## Fine-tuning $k_1$ and $b$ configs

### What is $k_1$? (The Repetition Ceiling) $k_1$ controls Term Frequency Saturation. It decides how much to reward a document for repeating the exact same keyword over and over.
- The Problem: If a document mentions "Metformin" 1 time, it is relevant. If it mentions it 3 times, it is probably very relevant. But if a massive glossary mentions it 50 times, is it 50 times more relevant? Probably not; it's just a long list.
- How $k_1$ fixes it: It sets a mathematical ceiling.
  - Low $k_1$ (e.g., 0.5): The score stops growing very quickly. The algorithm says, "Okay, you mentioned it twice, I get it. You don't get any more points for mentioning it 10 more times."
  - High $k_1$ (e.g., 2.5): The ceiling is very high. The algorithm continues to aggressively reward documents that hammer the keyword over and over.

### What is $b$? (The Long-Winded Penalty) $b$ controls Document Length Normalization. It decides how aggressively to penalize a document just for being long.
- The Problem: Imagine a user searches for "HbA1c". Document A is a tight, 2-sentence definition of HbA1c. Document B is a 50-page chapter on general diabetes care that happens to mention HbA1c once. If we don't penalize length, the 50-page document might win just by accident.
- How $b$ fixes it: It compares every document's length against the average length of the whole database.
  - $b = 1.0$ (Maximum Penalty): The algorithm acts like a strict editor. It heavily punishes long, rambling documents and forces short, hyper-focused paragraphs to the top of the search results.
  - $b = 0.0$ (Zero Penalty): The algorithm completely ignores how long the document is. A 5-page wall of text and a 1-sentence summary are graded on the exact same scale.

In [ ]:
import numpy as np

print("Building the 5 mathematical BM25 configurations with Ultimate Tokenizer...")

corpus_ultimate = [ultimate_tokenize(doc) for doc in all_doc_texts]

bm25_tuning = {
    "1. Baseline (k1=1.5, b=0.75)": BM25Okapi(corpus_ultimate, k1=1.5, b=0.75),
    "2. High k1 (k1=2.5, b=0.75)": BM25Okapi(corpus_ultimate, k1=2.5, b=0.75), # Rewards keyword repetition
    "3. Low k1 (k1=0.5, b=0.75)": BM25Okapi(corpus_ultimate, k1=0.5, b=0.75),  # Values presence over repetition
    "4. High b (k1=1.5, b=1.0)": BM25Okapi(corpus_ultimate, k1=1.5, b=1.0),   # Strictly penalizes long documents
    "5. Zero b (k1=1.5, b=0.0)": BM25Okapi(corpus_ultimate, k1=1.5, b=0.0)    # Ignores document length entirely
}

print("Indexes built! Running bake-off...\n")

prof_sample = retrieved_data_df[retrieved_data_df['Persona'] == 'Professional'].sample(n=5, random_state=42)
layperson_sample = retrieved_data_df[retrieved_data_df['Persona'] == 'Layperson'].sample(n=5, random_state=42)
test_batch = pd.concat([prof_sample, layperson_sample])

for _, row in test_batch.iterrows():
    persona = row['Persona']
    query_text = row['Question']

    print(f"\n{'='*90}")
    print(f"[{persona.upper()}] QUERY")
    print(f"Question: \"{query_text[:120]}...\"")
    print(f"{'='*90}")

    query_tokens = ultimate_tokenize(query_text)
    print(f"Tokens: {query_tokens}\n")

    for config_name, bm25_model in bm25_tuning.items():
        scores = bm25_model.get_scores(query_tokens)

        winner_idx = np.argsort(scores)[::-1][0]
        winning_score = scores[winner_idx]

        if winning_score > 0:
            snippet = all_doc_texts[winner_idx].replace('\n', ' ')[:1500]
            print(f"  {config_name:<28} | Score: {winning_score:.2f} | {snippet}...")
        else:
            print(f"  {config_name:<28} | NO MATCH FOUND")

Building the 5 mathematical BM25 configurations with Ultimate Tokenizer...
Indexes built! Running bake-off...


[PROFESSIONAL] QUERY
Question: "What are the primary tools used to assess glycemic status in a patient with diabetes?..."
Tokens: ['primary', 'tools', 'used', 'assess', 'glycemic', 'status', 'patient', 'diabetes']

  1. Baseline (k1=1.5, b=0.75) | Score: 14.88 | and follow-up, as appropriate, to: • Confirm the diagnosis and classify diabetes. A • Assess glycemic status and previous and current treatment. A • Evaluate for diabetes complications, potential comorbid conditions, and overall health status. A • Identify care partners, support systems, and available resources. E • Assess social determinants of health and structural barriers to optimal health and health care. A • Review risk factor management in the person with diabetes. A • Begin engagement wi...
  2. High k1 (k1=2.5, b=0.75)  | Score: 16.20 | and follow-up, as appropriate, to: • Confirm the diagnosis and classify d

In [ ]:
import pandas as pd
import numpy as np

# Collect results into a structured format
bakeoff_results = []

for _, row in test_batch.iterrows():
    persona = row['Persona']
    query_text = row['Question']
    query_tokens = ultimate_tokenize(query_text)

    row_data = {
        'Persona': persona,
        'Query': query_text[:30] + '...',
    }

    top_docs = {}
    for config_name, bm25_model in bm25_tuning.items():
        scores = bm25_model.get_scores(query_tokens)
        winner_idx = np.argsort(scores)[::-1][0]
        short_name = config_name.split('.')[1].strip().split('(')[0].strip()
        row_data[short_name] = round(scores[winner_idx], 2)
        top_docs[short_name] = winner_idx

    unique_tops = set(top_docs.values())
    if len(unique_tops) == 1:
        row_data['Same top doc?'] = 'All same'
    else:
        majority_doc = max(set(top_docs.values()), key=list(top_docs.values()).count)
        outliers = [name for name, idx in top_docs.items() if idx != majority_doc]
        row_data['Same top doc?'] = f"{5 - len(outliers)}/5 same ({', '.join(outliers)} different)"

    bakeoff_results.append(row_data)

comparison_df = pd.DataFrame(bakeoff_results)
display(comparison_df)

,Persona,Query,Baseline,High k1,Low k1,High b,Zero b,Same top doc?
0,Professional,What are the primary tools use...,14.88,16.20,13.16,14.89,14.87,4/5 same (Low k1 different)
1,Professional,When might a less stringent A1...,16.64,17.08,15.85,16.73,16.38,All same
2,Professional,"Mr. Tan, 52 years old, present...",54.03,56.00,50.85,53.46,55.81,All same
3,Professional,A patient with type 2 diabetes...,19.36,20.74,17.17,19.25,19.69,All same
4,Professional,A newly diagnosed type 2 diabe...,30.92,34.02,27.19,30.53,32.17,4/5 same (Low k1 different)
5,Layperson,"""My HbA1c has improved but my ...",12.47,13.25,11.43,12.38,12.77,4/5 same (Low k1 different)
6,Layperson,"""My doctor wants me to wear a ...",11.59,12.53,11.59,11.64,12.19,"3/5 same (High k1, High b different)"
7,Layperson,"""I am going in for surgery nex...",14.23,15.00,12.86,14.37,13.83,4/5 same (High k1 different)
8,Layperson,"""My father is very ill with ca...",26.14,27.04,24.50,26.27,25.74,All same
9,Layperson,"""I cannot afford the glucose s...",10.63,12.53,8.33,10.64,10.89,"3/5 same (Low k1, Zero b different)"


In [ ]:
import numpy as np

# ==========================================
# 3. EVALUATE BM25 CONFIGS WITH run_retrieval
# ==========================================
print("Running BM25 Config Evaluation with RAGAS Context Precision...\n")

bm25_results = {}

for config_name, bm25_model in bm25_tuning.items():
    print(f"Evaluating: {config_name}...")

    # Retrieve top-k docs for each query using this BM25 config
    retrieved_contexts_col = []
    for _, row in test_queries.iterrows():
        query_tokens = ultimate_tokenize(row['Question'])
        scores = bm25_model.get_scores(query_tokens)
        top_indices = np.argsort(scores)[::-1][:5]
        top_chunks = [all_doc_texts[i] for i in top_indices if scores[i] > 0]
        retrieved_contexts_col.append(top_chunks)

    # Build the DataFrame in the format run_retrieval expects
    eval_df = test_queries[['Persona', 'Question', 'Reference']].copy()
    eval_df['Retrieved Contexts'] = retrieved_contexts_col

    # Run RAGAS Context Precision
    precision_scores = run_retrieval(eval_df)

    # Store results with persona info
    eval_df['context_precision'] = precision_scores.values
    bm25_results[config_name] = eval_df[['Persona', 'Question', 'context_precision']].copy()
    bm25_results[config_name]['config'] = config_name

    avg = eval_df['context_precision'].mean()
    print(f"  -> Average Context Precision: {avg:.4f}\n")

# ==========================================
# 4. COMBINE & DISPLAY RESULTS
# ==========================================
all_results_df = pd.concat(bm25_results.values(), ignore_index=True)

# Per-config summary
summary = all_results_df.groupby(['config', 'Persona'])['context_precision'].mean().unstack()
print("CONTEXT PRECISION BY CONFIG AND PERSONA\n")
display(summary)

# Overall leaderboard
overall = all_results_df.groupby('config')['context_precision'].mean().sort_values(ascending=False)
print("\nOVERALL LEADERBOARD\n")
display(overall.to_frame('Avg Context Precision'))

# Per-query breakdown for the top config
best_config = overall.index[0]
print(f"\nDETAILED RESULTS FOR BEST CONFIG: {best_config}\n")
best_df = all_results_df[all_results_df['config'] == best_config].copy()
best_df['Question'] = best_df['Question'].str[:80] + '...'
display(best_df[['Persona', 'Question', 'context_precision']])

Running BM25 Config Evaluation with RAGAS Context Precision...

Evaluating: 1. Baseline (k1=1.5, b=0.75)...


Evaluating:   0%|          | 0/10 [00:00<?, ?it/s]

  -> Average Context Precision: 0.7040

Evaluating: 2. High k1 (k1=2.5, b=0.75)...


Evaluating:   0%|          | 0/10 [00:00<?, ?it/s]

  -> Average Context Precision: 0.6804

Evaluating: 3. Low k1 (k1=0.5, b=0.75)...


...

Persona,Layperson,Professional
config,,
"1. Baseline (k1=1.5, b=0.75)",0.573056,0.835000
"2. High k1 (k1=2.5, b=0.75)",0.586667,0.774167
"3. Low k1 (k1=0.5, b=0.75)",0.532222,0.940833
"4. High b (k1=1.5, b=1.0)",0.531389,0.839167
"5. Zero b (k1=1.5, b=0.0)",0.660833,0.787778



OVERALL LEADERBOARD



,Avg Context Precision
config,
"3. Low k1 (k1=0.5, b=0.75)",0.736528
"5. Zero b (k1=1.5, b=0.0)",0.724306
"1. Baseline (k1=1.5, b=0.75)",0.704028
"4. High b (k1=1.5, b=1.0)",0.685278
"2. High k1 (k1=2.5, b=0.75)",0.680417



DETAILED RESULTS FOR BEST CONFIG: 3. Low k1 (k1=0.5, b=0.75)



,Persona,Question,context_precision
20,Professional,What are the primary tools used to assess glyc...,0.804167
21,Professional,When might a less stringent A1C goal be approp...,1.000000
22,Professional,"Mr. Tan, 52 years old, presents to your clinic...",0.950000
23,Professional,A patient with type 2 diabetes on an SGLT2 inh...,0.950000
24,Professional,A newly diagnosed type 2 diabetes patient has ...,1.000000
25,Layperson,"""My HbA1c has improved but my sensor shows I a...",0.000000
26,Layperson,"""My doctor wants me to wear a glucose sensor o...",0.866667
27,Layperson,"""I am going in for surgery next week. Do I nee...",0.477778
28,Layperson,"""My father is very ill with cancer and diabete...",0.950000
29,Layperson,"""I cannot afford the glucose sensor. What can ...",0.366667


### BM25 Parameter Tuning Findings

We tested five configurations of k1 and b across 10 queries (5 Professional, 5 Layperson) using our ultimate tokenizer. We evaluated using both raw BM25 relevance scores and RAGAS Context Precision (LLM-judged retrieval quality against golden reference chunks).

Key insight: Raw retrieval scores alone are insufficient for evaluating BM25 configurations. High k1 consistently produced the highest raw scores and all configurations mostly retrieved the same top document. This initially suggested that parameter tuning didn't matter. However, RAGAS Context Precision which measures whether the retrieved chunks actually contain the right information  revealed meaningful differences in retrieval quality.

k1 (term frequency saturation):

- Low k1 (0.5) achieved the highest overall Context Precision (0.736) and dominated on Professional queries (0.940 vs Baseline's 0.836). By capping the reward for repeated terms, it favors chunks that contain a wider spread of query keywords. For multi-concept professional queries like "SGLT2 inhibitor + elective surgery + medication management," this retrieves chunks that cover all aspects rather than chunks that simply repeat "diabetes" many times.
- High k1 (2.5) performed better on Layperson queries (0.586) than the baseline (0.5731) and low k1 (0.5322). This is intereting and a bit counter-intuitive. For Layperson queries, over-rewarding term repetition pulls toward keyword-dense chunks, but chunks can sometimes be topically mismatched. But in our case, those chunks happen to be the correct ones.

b (document length normalization):

- Zero b (0.0) performed second-best overall (0.724) and led on Layperson queries (0.660). By ignoring document length, it avoids penalizing shorter, more focused chunks that may contain exactly the answer a layperson needs.
- High b (1.0) performed second worst overall (0.685). Our VDB chunks are roughly uniform in length, so aggressive length penalization provides no benefit and slightly degrades results.

Best config: Low k1 (k1=0.5, b=0.75). It provides the highest overall Context Precision (0.737), outperforms Baseline on Professional queries, with only a marginal difference on Layperson queries. The tradeoff is clearly favorable.

# Hybrid Search using Ensemble Retriever

### We will use **Reciprocal Rank Fusion** (RRF) to combine the semantic search (dense) and the BM25 keyword search (sparse).

**Reciprocal Rank Fusion** combines rankings from multiple retrievers. For each document, the RRF score is:

$$\text{RRF}(d) = \sum_{r \in \text{retrievers}} \frac{1}{k + \text{rank}_r(d)}$$

where $k$ is a constant (typically 60) that dampens the influence of high-ranking differences. Documents appearing in **both** lists get a boost.

## Testing different retrieval weights

In [ ]:
from langchain_classic.retrievers.ensemble import EnsembleRetriever

# ==========================================
# 1. Initialize the Semantic Retriever
# ==========================================
# We use the 0.6 threshold
semantic_retriever = vectorstore.as_retriever(
    search_type="similarity_score_threshold",
    search_kwargs={
        "score_threshold": 0.6,
        "k": 5
    }
)

# ==========================================
# 2. Initialize fine-tuned BM25 Retriever
# ==========================================
# We use the ultimate_tokenize logic and the best-performing config k1=0.5, b=0.75)
from langchain_community.retrievers import BM25Retriever

# LangChain's wrapper will build the Okapi index under the hood,
# but we are forcing it to use the custom smart tokenizer and best config.
bm25_retriever = BM25Retriever.from_texts(
    texts=all_doc_texts,
    metadatas=all_doc_metadatas,
    preprocess_func=ultimate_tokenize,  # use smart tokenizer
    bm25_params={"k1": 0.5, "b": 0.75}  # use winning config
)

bm25_retriever.k = 5

In [ ]:
import random
random.seed(42)

print("Building Hybrid Retrievers with different RRF weights...")

weight_configs = {
    "1. Vector Dominant (80/20)": [0.8, 0.2],
    "2. Vector Leaning (60/40)": [0.6, 0.4],
    "3. Balanced (50/50)": [0.5, 0.5],
    "4. Keyword Leaning (40/60)": [0.4, 0.6],
    "5. Keyword Dominant (20/80)": [0.2, 0.8],
}

hybrid_retrievers = {}
for config_name, weights in weight_configs.items():
    hybrid_retrievers[config_name] = EnsembleRetriever(
        retrievers=[semantic_retriever, bm25_retriever],
        weights=weights
    )

print("Hybrid Retrievers built! Running evaluation...\n")

prof_sample = retrieved_data_df[retrieved_data_df['Persona'] == 'Professional'].sample(n=5, random_state=42)
layperson_sample = retrieved_data_df[retrieved_data_df['Persona'] == 'Layperson'].sample(n=5, random_state=42)
test_batch = pd.concat([prof_sample, layperson_sample])

for _, row in test_batch.iterrows():
    persona = row['Persona']
    query_text = row['Question']

    print(f"\n{'='*90}")
    print(f"[{persona.upper()}] QUERY")
    print(f"Question: \"{query_text[:1200]}...\"")
    print(f"{'='*90}")

    for config_name, retriever in hybrid_retrievers.items():
        try:
            docs = retriever.invoke(query_text)
            if docs:
                top_doc = docs[0].page_content.replace('\n', ' ')[:1500]
                print(f"  {config_name:<30} | {top_doc}...")
            else:
                print(f"  {config_name:<30} | NO DOCUMENTS FOUND")
        except Exception as e:
            print(f"  {config_name:<30} | ERROR: {e}")

Building Hybrid Retrievers with different RRF weights...
Hybrid Retrievers built! Running evaluation...


[PROFESSIONAL] QUERY
Question: "What are the primary tools used to assess glycemic status in a patient with diabetes?..."
  1. Vector Dominant (80/20)     | cemic status should be used, including self-monitoring of blood glucose, CGM, and/or the use of glycated serum protein assays (discussed below). A1C does not provide a measure of glycemic variability, real-time glucose levels, or hypoglycemia. For individuals prone to glycemic variability, especially people with type 1 diabetes or type 2 diabetes with insulin deficiency and/or treatment with intensive insulin therapy, glycemic status is best evaluated by the combination of results from BGM or CG...
  2. Vector Leaning (60/40)      | and follow-up, as appropriate, to: • Confirm the diagnosis and classify diabetes. A • Assess glycemic status and previous and current treatment. A • Evaluate for diabetes complications, potential co

In [ ]:
import pandas as pd

# ==========================================
# Hybrid Weight Config Comparison Table
# ==========================================

comparison_results = []

for _, row in test_batch.iterrows():
    persona = row['Persona']
    query_text = row['Question']

    top_docs = {}
    for config_name, retriever in hybrid_retrievers.items():
        docs = retriever.invoke(query_text)
        if docs:
            top_docs[config_name] = docs[0].page_content[:1000]
        else:
            top_docs[config_name] = "NO DOCS"

    # Check agreement: compare each config's top chunk to the majority
    chunk_list = list(top_docs.values())
    majority_chunk = max(set(chunk_list), key=chunk_list.count)
    agree_count = chunk_list.count(majority_chunk)

    if agree_count == len(chunk_list):
        agreement = "All same"
    else:
        short_names = {
            "1. Vector Dominant (80/20)": "80/20",
            "2. Vector Leaning (60/40)": "60/40",
            "3. Balanced (50/50)": "50/50",
            "4. Keyword Leaning (40/60)": "40/60",
            "5. Keyword Dominant (20/80)": "20/80",
        }
        outliers = [
            short_names.get(name, name)
            for name, chunk in top_docs.items()
            if chunk != majority_chunk
        ]
        agreement = f"{agree_count}/{len(chunk_list)} same ({', '.join(outliers)} different)"

    comparison_results.append({
        'Persona': persona,
        'Query': query_text[:400] + '...',
        'Agreement': agreement,
    })

comparison_df = pd.DataFrame(comparison_results)
display(comparison_df)

,Persona,Query,Agreement
0,Professional,What are the primary tools used to assess glyc...,4/5 same (80/20 different)
1,Professional,When might a less stringent A1C goal be approp...,All same
2,Professional,"Mr. Tan, 52 years old, presents to your clinic...","3/5 same (40/60, 20/80 different)"
3,Professional,A patient with type 2 diabetes on an SGLT2 inh...,All same
4,Professional,A newly diagnosed type 2 diabetes patient has ...,All same
5,Layperson,"""My HbA1c has improved but my sensor shows I a...",All same
6,Layperson,"""My doctor wants me to wear a glucose sensor o...","3/5 same (40/60, 20/80 different)"
7,Layperson,"""I am going in for surgery next week. Do I nee...",All same
8,Layperson,"""My father is very ill with cancer and diabete...",All same
9,Layperson,"""I cannot afford the glucose sensor. What can ...","3/5 same (40/60, 20/80 different)"


#### From our initial inspection, 6 out of 10 queries have all 5 configs retrieving the same top chunk.
The diverged queries are looked closer below:

|Query|Persona|Split point|Vector-leaning retrieves|Keyword-leaning retrieves|Better?|
|---|---|---|---|---|---|
|Glycemic status tools|Professional|80/20 vs rest|Lists specific tools: BGM, CGM, glycated serum protein assays|General clinical checklist ("Assess glycemic status...")|80/20|
|Mr. Tan vignette|Professional|50/50 vs 40/60|Lab reference table (UACR, eGFR, CBC, FIB-4)|Generic hypoglycemia classification|Vector-leaning|
|Glucose sensor on arm|Layperson|50/50 vs 40/60|CGM education and training guidance|CGM accuracy statistics (%20/20 agreement rates)|Vector-leaning|
|Can't afford sensor|Layperson|50/50 vs 40/60|Hypoglycemia symptoms with CGM (wrong but on-topic)|Early pregnancy treatment -- matched "afford" to "afford incremental benefit"|Vector-leaning|
#### All 4 divergences split in the same direction: keyword-leaning configs (40/60, 20/80) degrade quality once they cross the 50% BM25 threshold. The split always occurs between Balanced (50/50) and Keyword Leaning (40/60).


#### The first query (glycemic status tools) is the only one where 80/20 retrieves something better than from 60/40. It finds the chunk that actually directly mentions the tools. But 60/40 still retrieves a reasonable chunk (clinical checklist mentioning glycemic assessment), while the keyword-dominant configs always retrieve something worse or irrelevant.

#### The 60/40 Vector Leaning seems the safest choice so far. It matches the best result on 9 out of 10 queries and only narrowly loses to 80/20 on one query.

In [ ]:
# ==========================================
# Evaluate Hybrid Weight Configs with run_retrieval
# ==========================================
print("Running Hybrid Weight Config Evaluation with RAGAS Context Precision...\n")

hybrid_weight_results = {}

for config_name, retriever in hybrid_retrievers.items():
    print(f"Evaluating: {config_name}...")

    retrieved_contexts_col = []
    for _, row in test_queries.iterrows():
        docs = retriever.invoke(row['Question'])
        retrieved_contexts_col.append([doc.page_content for doc in docs])

    eval_df = test_queries[['Persona', 'Question', 'Reference']].copy()
    eval_df['Retrieved Contexts'] = retrieved_contexts_col

    precision_scores = run_retrieval(eval_df)

    eval_df['context_precision'] = precision_scores.values
    hybrid_weight_results[config_name] = eval_df[['Persona', 'Question', 'context_precision']].copy()
    hybrid_weight_results[config_name]['config'] = config_name

    avg = eval_df['context_precision'].mean()
    print(f"  -> Average Context Precision: {avg:.4f}\n")

# ==========================================
# Combine & Display Results
# ==========================================
all_hybrid_df = pd.concat(hybrid_weight_results.values(), ignore_index=True)

summary = all_hybrid_df.groupby(['config', 'Persona'])['context_precision'].mean().unstack()
print("CONTEXT PRECISION BY CONFIG AND PERSONA\n")
display(summary)

overall = all_hybrid_df.groupby('config')['context_precision'].mean().sort_values(ascending=False)
print("\nOVERALL LEADERBOARD\n")
display(overall.to_frame('Avg Context Precision'))

best_config = overall.index[0]
print(f"\nDETAILED RESULTS FOR BEST CONFIG: {best_config}\n")
best_df = all_hybrid_df[all_hybrid_df['config'] == best_config].copy()
best_df['Question'] = best_df['Question'].str[:80] + '...'
display(best_df[['Persona', 'Question', 'context_precision']])

Running Hybrid Weight Config Evaluation with RAGAS Context Precision...

Evaluating: 1. Vector Dominant (80/20)...


Evaluating:   0%|          | 0/10 [00:00<?, ?it/s]

  -> Average Context Precision: 0.8206

Evaluating: 2. Vector Leaning (60/40)...


Evaluating:   0%|          | 0/10 [00:00<?, ?it/s]

  -> Average Context Precision: 0.8043

Evaluating: 3. Balanced (50/50)...


...

Persona,Layperson,Professional
config,,
1. Vector Dominant (80/20),0.749683,0.891432
2. Vector Leaning (60/40),0.749683,0.858893
3. Balanced (50/50),0.692946,0.829152
4. Keyword Leaning (40/60),0.615776,0.872163
5. Keyword Dominant (20/80),0.668579,0.817720



OVERALL LEADERBOARD



,Avg Context Precision
config,
1. Vector Dominant (80/20),0.820557
2. Vector Leaning (60/40),0.804288
3. Balanced (50/50),0.761049
4. Keyword Leaning (40/60),0.743969
5. Keyword Dominant (20/80),0.743149



DETAILED RESULTS FOR BEST CONFIG: 1. Vector Dominant (80/20)



,Persona,Question,context_precision
0,Professional,What are the primary tools used to assess glyc...,0.810000
1,Professional,When might a less stringent A1C goal be approp...,0.873639
2,Professional,"Mr. Tan, 52 years old, presents to your clinic...",0.880612
3,Professional,A patient with type 2 diabetes on an SGLT2 inh...,0.986111
4,Professional,A newly diagnosed type 2 diabetes patient has ...,0.906796
5,Layperson,"""My HbA1c has improved but my sensor shows I a...",0.679167
6,Layperson,"""My doctor wants me to wear a glucose sensor o...",0.942857
7,Layperson,"""I am going in for surgery next week. Do I nee...",0.848611
8,Layperson,"""My father is very ill with cancer and diabete...",0.986111
9,Layperson,"""I cannot afford the glucose sensor. What can ...",0.291667


### Counter-intuitive Results
- We eyeballed the top-1 document only and concluded vector-leaning configs were better. But RAGAS Context Precision evaluates all retrieved chunks (positions 1 through k). A config can have a mediocre top-1 result but pack more relevant material into positions 2-5.

- BM25 might put a noisy chunk at rank 1 (like the hypoglycemia classification for Mr. Tan), but it also pulls in several highly relevant chunks at ranks 2-4 that the vector retriever misses entirely.

### Compatible Config for Professional Persona
- Professional queries prefer more BM25. Keyword Dominant leads at 0.838. Clinical queries contain specific terms (SGLT2, eGFR, A1C) that BM25 matches precisely. Even when the top-1 is wrong, BM25 retrieves more total relevant chunks because medical guidelines repeat these terms in many relevant sections.

- Layperson queries are not performing well. No config solves layperson retrieval well. The "can't afford sensor" query scores 0.25 across all configs because the VDB simply doesn't contain affordability content.

### Key Insight: What matters is the total relevant material in the retrieved set.

## Hybrid Search Weight Config Conclusion
### Vector Dominant (80/20) is the best overall config. This proves that for our vector store semantic search plays a more significnat role in identifying the correct chunk.

#### We also suspect these lower context precision and generally worse performance in "Layperson" queries may be resulted from the ambiguous terms used in the queries. To address the issue of poor wordings in those queries, we can look at RAG Fusion, one of the query expansion techniques that addresses the issue of ambiguous user queries by rewriting them into multiple search variations.

# RAG Fusion (Multi-Query Variants + RRF)
We are shifting our focus from the retrieval math to the linguistic mapping. Because we are dealing with medical data, the goal is to bridge the gap between a patient's conversational phrasing and the textbook's clinical phrasing.

RAG Fusion generates N **different sub-queries** from different perspectives on the original question, retrieves documents for each sub-query independently, then merges all result lists using **Reciprocal Rank Fusion (RRF)**.

This casts a wider retrieval net than a single query.

To optimize this linguistic bridge, we are evaluating the following key tunable parameters:

1. **The LLM Generator (Model):** The "brain" rewriting the queries. We test different weight classes (e.g., `llama3.1`, `phi4-mini`) to balance deep clinical vocabulary against generation speed and computational cost.
    
2. **Prompt Strategy:** How we instruct the LLM to rewrite the query.
    
    - _Multi-Angle:_ Asks the LLM to explore different facets of the question (e.g., symptoms, treatments, mechanisms).
        
    - _Clinical Translation:_ Strictly forces the LLM to convert layman terms into exact medical jargon and pharmacological names.
        
3. **`NUM_QUERIES` ($N$):** How many sub-queries to generate. A higher number casts a wider net, but generating too many risks "topic drift" where the LLM starts hallucinating unrelated search terms.
        

_(Note: Temperature is intentionally fixed at `0.1` for this step to provide just enough micro-variance to prevent identical query generation, without allowing the LLM to hallucinate false medical conditions)._

### Setup Ollama LLM for query rewriting

In [ ]:
# Install Ollama
%%capture
!sudo apt update -qq
!sudo apt install -y -qq pciutils
!sudo apt-get install -y -qq zstd
!curl -fsSL https://ollama.com/install.sh | sh


# Start Ollama server
import threading
import subprocess
import time
import os

def run_ollama_serve():
    subprocess.Popen(["ollama", "serve"])

thread = threading.Thread(target=run_ollama_serve)
thread.start()
time.sleep(5)
print("✅ Ollama server started.")


# Pull model
!ollama pull llama3.1
!ollama pull phi4-mini
!ollama pull mistral:7b
!ollama list

## Set up fusion prompts and retrievers

In [ ]:
fusion_prompts = {
    "Multi-Angle": """You are an expert clinical research assistant. Your goal is to retrieve relevant medical guidelines to answer the user's question.
Generate exactly {num_queries} distinct search queries that explore this topic from different clinical domains (e.g., diagnostic criteria, pharmacological management, pathophysiology, lifestyle interventions).

CRITICAL INSTRUCTIONS:
- Write concise, keyword-rich search phrases (e.g., "type 2 diabetes metformin contraindications").
- DO NOT write full interrogative questions (Avoid "What are the...").
- Output ONLY the raw queries, one per line. Absolutely NO numbers, NO bullet points, NO quotes, and NO introductory text.

Original Question: {question}""",

    "Clinical_Translation": """You are a senior medical terminologist. Translate the user's question into exactly {num_queries} distinct, highly precise clinical search queries.

CRITICAL INSTRUCTIONS:
- Convert layman terminology into standard clinical jargon (e.g., "weak heart" -> "congestive heart failure etiology").
- Stick to terminology used in standard medical guidelines and broad pharmacological classes. Avoid ultra-obscure jargon.
- Write concise, keyword-dense search phrases, NOT conversational sentences.
- Output ONLY the raw queries, one per line. Absolutely NO numbers, NO bullet points, NO quotes, and NO conversational filler.

Original Question: {question}"""
}


RRF_K = 60
FUSION_TOP_K = 5
TEMPERATURE = 0.1

In [ ]:
# semantic_retriever = vectorstore.as_retriever(
#     search_type="similarity_score_threshold",
#     search_kwargs={
#         "score_threshold": 0.6,
#         "k": 5
#     }
# )


best_hybrid_retriever = hybrid_retrievers["1. Vector Dominant (80/20)"]

print(f"Base Hybrid Retriever: Vector Dominant (80/20)")
print(f"Retrieval Depth: K={FUSION_TOP_K} | RRF_K={RRF_K} | Temp={TEMPERATURE}")


Base Hybrid Retriever: Vector Dominant (80/20)
Retrieval Depth: K=5 | RRF_K=60 | Temp=0.1


In [ ]:
import pandas as pd
import time
from langchain_ollama import ChatOllama
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

print("🔍 Initializing the Query Inspector...\n")

# ==========================================
# 1. GRAB TWO SPECIFIC QUESTIONS (Seed = 42)
# ==========================================
# Filter the dataframe for the specific personas, then sample 1 of each
layman_q = retrieved_data_df[retrieved_data_df['Persona'] == 'Layperson'].sample(n=1, random_state=42)['Question'].iloc[0]
professional_q = retrieved_data_df[retrieved_data_df['Persona'] == 'Professional'].sample(n=1, random_state=42)['Question'].iloc[0]

questions_to_test = {
    "Layperson": layman_q,
    "Professional": professional_q
}

# ==========================================
# 2. Using Multi-Angle Fusion Prompt
# ==========================================
prompt_text = """You are an expert clinical research assistant. Your goal is to retrieve relevant medical guidelines to answer the user's question.
Generate exactly {num_queries} distinct search queries that explore this topic from different clinical domains (e.g., diagnostic criteria, pharmacological management, pathophysiology, lifestyle interventions).

CRITICAL INSTRUCTIONS:
- Write concise, keyword-rich search phrases (e.g., "type 2 diabetes metformin contraindications").
- DO NOT write full interrogative questions (Avoid "What are the...").
- Output ONLY the raw queries, one per line. Absolutely NO numbers, NO bullet points, NO quotes, and NO introductory text.

Original Question: {question}"""

prompt_template = ChatPromptTemplate.from_template(prompt_text)


def parse_sub_queries(output):
    lines = [q.strip().strip('"').strip("'").lstrip("1234567890.- ") for q in output.split("\n") if q.strip()]
    return [q for q in lines if len(q) > 5]

# ==========================================
# 3. INSPECTION FUNCTION
# ==========================================
def inspect_generation(model_tag, question_dict):
    print(f"\n{'='*60}")
    print(f"🧠 MODEL EVALUATION: {model_tag.upper()}")
    print(f"{'='*60}")

    llm = ChatOllama(model=model_tag, temperature=0.1)
    chain = prompt_template | llm | StrOutputParser() | parse_sub_queries

    for q_type, q_text in question_dict.items():
        print(f"\n🧑‍⚕️ {q_type.upper()} QUESTION: '{q_text}'")
        print("-" * 60)

        try:
            queries = chain.invoke({"num_queries": 5, "question": q_text})
            for i, q in enumerate(queries, 1):
                print(f"  {i}. {q}")
        except Exception as e:
            print(f"  ⚠️ Error generating queries: {e}")

# ==========================================
# 4. RUN THE COMPARISON
# ==========================================
# Test how the mini model compares to the industry baseline
inspect_generation("phi4-mini", questions_to_test)
inspect_generation("llama3.1", questions_to_test)

🔍 Initializing the Query Inspector...


🧠 MODEL EVALUATION: PHI4-MINI

🧑‍⚕️ LAYPERSON QUESTION: '"My HbA1c has improved but my sensor shows I am having low blood sugars at night. Which result should I trust?"'
------------------------------------------------------------
  1. type 2 diabetes continuous glucose monitoring vs hemoglobin A1c
  2. continuous glucose monitor (CGM) accuracy compared to fasting plasma glucose test in type 2 diabetics
  3. HbA1c and nocturnal hypoglycemia: clinical implications for diabetic patients using CGMs
  4. interpreting discrepancies between HbA1c levels, blood sugar readings from a sensor/glucose meter at night time.
  5. pharmacological management of diabetes with continuous monitoring systems showing low nighttime glucose.

🧑‍⚕️ PROFESSIONAL QUESTION: 'What are the primary tools used to assess glycemic status in a patient with diabetes?'
------------------------------------------------------------
  1. glycemic control assessment methods
  2. HbA1c m

### Initial findings on "Multi-Angle" prompt

1. Phi4-mini:

  - phi4-mini has an instruction-following failure on the professional query. It generated 6 sub-queries when asked for 5. Worse, query 6 is outright hallucination:

    - "Primary Tools Used to Assess Glycemic Status in Diabetes: Metformin Contraindications, Insulin Therapy Considerations, Sulfonylureas Side Effects, DPP-4 Inhibitors Risks and Benefits."
- phi4-mini generates verbose, sentence-like queries:

     - "interpreting discrepancies between HbA1c levels, blood sugar readings from a sensor/glucose meter at night time."

2. Llama3.1
- llama3.1 produced exactly 5 clean, on-target queries for the same question. No drift.

- llama3.1 generates tighter, keyword-dense phrases:

  - "Diabetes self-management with conflicting HbA1c and CGM results"

3. Layperson vs Professional

- On the layperson query, both are solid. Both successfully translated "low blood sugars at night" into "nocturnal hypoglycemia" and "sensor" into "CGM/continuous glucose monitoring." phi4-mini's query 5 drifts into "pharmacological management" (the patient asked which result to trust, not about medication changes), but it's a minor issue.

For retrieval, the shorter keyword-dense style tends to work better because it matches how medical textbooks are written. Long conversational queries dilute the BM25 signal with filler words. llama3.1 seems to be the more disciplined model -- cleaner format, strict instruction following, no hallucination.

In [ ]:
import pandas as pd
import time
from langchain_ollama import ChatOllama
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

print("🔍 Initializing the Query Inspector...\n")

# ==========================================
# 1. GRAB TWO SPECIFIC QUESTIONS (Seed = 42)
# ==========================================
# Filter the dataframe for the specific personas, then sample 1 of each
layman_q = retrieved_data_df[retrieved_data_df['Persona'] == 'Layperson'].sample(n=1, random_state=42)['Question'].iloc[0]
professional_q = retrieved_data_df[retrieved_data_df['Persona'] == 'Professional'].sample(n=1, random_state=42)['Question'].iloc[0]

questions_to_test = {
    "Layperson": layman_q,
    "Professional": professional_q
}

# ==========================================
# 2. Using Clinical_Translation Fusion Prompt
# ==========================================
prompt_text = """You are a senior medical terminologist. Translate the user's question into exactly {num_queries} distinct, highly precise clinical search queries.

CRITICAL INSTRUCTIONS:
- Convert layman terminology into standard clinical jargon (e.g., "weak heart" -> "congestive heart failure etiology").
- Stick to terminology used in standard medical guidelines and broad pharmacological classes. Avoid ultra-obscure jargon.
- Write concise, keyword-dense search phrases, NOT conversational sentences.
- Output ONLY the raw queries, one per line. Absolutely NO numbers, NO bullet points, NO quotes, and NO conversational filler.

Original Question: {question}"""

prompt_template = ChatPromptTemplate.from_template(prompt_text)


def parse_sub_queries(output):
    lines = [q.strip().strip('"').strip("'").lstrip("1234567890.- ") for q in output.split("\n") if q.strip()]
    return [q for q in lines if len(q) > 5]

# ==========================================
# 3. INSPECTION FUNCTION
# ==========================================
def inspect_generation(model_tag, question_dict):
    print(f"\n{'='*60}")
    print(f"🧠 MODEL EVALUATION: {model_tag.upper()}")
    print(f"{'='*60}")

    llm = ChatOllama(model=model_tag, temperature=0.1)
    chain = prompt_template | llm | StrOutputParser() | parse_sub_queries

    for q_type, q_text in question_dict.items():
        print(f"\n🧑‍⚕️ {q_type.upper()} QUESTION: '{q_text}'")
        print("-" * 60)

        try:
            queries = chain.invoke({"num_queries": 5, "question": q_text})
            for i, q in enumerate(queries, 1):
                print(f"  {i}. {q}")
        except Exception as e:
            print(f"  ⚠️ Error generating queries: {e}")

# ==========================================
# 4. RUN THE COMPARISON
# ==========================================
# Test how the mini model compares to the industry baseline
inspect_generation("phi4-mini", questions_to_test)
inspect_generation("llama3.1", questions_to_test)

🔍 Initializing the Query Inspector...


🧠 MODEL EVALUATION: PHI4-MINI

🧑‍⚕️ LAYPERSON QUESTION: '"My HbA1c has improved but my sensor shows I am having low blood sugars at night. Which result should I trust?"'
------------------------------------------------------------
  1. HbA1c level improvement
  2. Nighttime hypoglycemia detection accuracy
  3. Hemoglobin A1c vs Continuous Glucose Monitoring reliability comparison
  4. Postprandial glucose levels versus Hemoglobin A1c for diabetes management assessment
  5. Continuous Blood Sugar Monitor HbA1c discrepancies analysis
  6. Diabetes Mellitus Type 2: Glycemic Control Evaluation using HbA1c and CGM Data Integration

🧑‍⚕️ PROFESSIONAL QUESTION: 'What are the primary tools used to assess glycemic status in a patient with diabetes?'
------------------------------------------------------------
  1. HbA1c measurement for diabetic control assessment
  2. Fasting plasma glucose test as an indicator of blood sugar levels
  3. Oral Glucose Tole

### Initial findings on "Clinical translation" prompt
1. Llama3.1:

- llama3.1 leaks a preamble into its output. Both queries start with:

  - "Here are 5 distinct clinical search queries:"
  - The parse_sub_queries function strips numbers and bullets, but the leaked generated sentence is longer than 5 characters and passes the filter. It becomes a real search query sent to the retriever, which would retrieve random irrelevant chunks. So llama3.1 is effectively generating 5 useful queries + 1 garbage query = 6 items, where the first is noise.

2. Phi4-mini:
- Despite generating a 6th sub query again, phi4-mini overall improved with Clinical Translation fusion prompt. The filler words are almost all gone.

3. Layperson vs Professional

- There is no meaningful quality gap in both models for Professional queries.
- llama3.1 (ignoring the preamble leak) produces tighter clinical mappings for Layperson queries:

  - "CGMS vs. SMBG for detecting nighttime hypoglycemic episodes" -- directly frames the patient's "which should I trust?" as a clinical comparison
  - "nocturnal glycemic variability" -- precise term for what the patient is experiencing

- phi4-mini consistently performs poorly in Layperson:

  - Query 1 ("HbA1c level improvement") is too vague -- it could retrieve anything about A1c trends, not specifically the sensor-vs-lab conflict
  - Query 4 ("Postprandial glucose levels versus Hemoglobin A1c") introduces postprandial glucose, which the patient never mentioned -- they said nighttime, not after meals

In [ ]:
import pandas as pd
import time
from langchain_ollama import ChatOllama
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

print("🔍 Initializing the Query Inspector...\n")

# ==========================================
# 1. GRAB TWO SPECIFIC QUESTIONS (Seed = 42)
# ==========================================
# Filter the dataframe for the specific personas, then sample 1 of each
layman_q = retrieved_data_df[retrieved_data_df['Persona'] == 'Layperson'].sample(n=1, random_state=42)['Question'].iloc[0]
professional_q = retrieved_data_df[retrieved_data_df['Persona'] == 'Professional'].sample(n=1, random_state=42)['Question'].iloc[0]

questions_to_test = {
    "Layperson": layman_q,
    "Professional": professional_q
}

# ==========================================
# 2. Using Clinical_Translation Fusion Prompt
# ==========================================
prompt_text = """You are a senior medical terminologist. Translate the user's question into exactly {num_queries} distinct, highly precise clinical search queries.

CRITICAL INSTRUCTIONS:
- Convert layman terminology into standard clinical jargon (e.g., "weak heart" -> "congestive heart failure etiology").
- Stick to terminology used in standard medical guidelines and broad pharmacological classes. Avoid ultra-obscure jargon.
- Write concise, keyword-dense search phrases, NOT conversational sentences.
- Output ONLY the raw queries, one per line. Absolutely NO numbers, NO bullet points, NO quotes, and NO conversational filler.

Original Question: {question}"""

prompt_template = ChatPromptTemplate.from_template(prompt_text)


def parse_sub_queries(output, max_queries=None):
    PREAMBLE_KEYWORDS = {"here are", "search queries", "following", "below are", "queries:"}
    lines = [q.strip().strip('"').strip("'").lstrip("1234567890.- ") for q in output.split("\n") if q.strip()]
    filtered = [
        q for q in lines
        if len(q) > 5 and not any(kw in q.lower() for kw in PREAMBLE_KEYWORDS)
    ]
    if max_queries:
        return filtered[:max_queries]
    return filtered

# ==========================================
# 3. INSPECTION FUNCTION
# ==========================================
def inspect_generation(model_tag, question_dict):
    print(f"\n{'='*60}")
    print(f"🧠 MODEL EVALUATION: {model_tag.upper()}")
    print(f"{'='*60}")

    llm = ChatOllama(model=model_tag, temperature=0.1)
    chain = prompt_template | llm | StrOutputParser() | parse_sub_queries

    for q_type, q_text in question_dict.items():
        print(f"\n🧑‍⚕️ {q_type.upper()} QUESTION: '{q_text}'")
        print("-" * 60)

        try:
            queries = chain.invoke({"num_queries": 5, "question": q_text})
            for i, q in enumerate(queries, 1):
                print(f"  {i}. {q}")
        except Exception as e:
            print(f"  ⚠️ Error generating queries: {e}")

# ==========================================
# 4. RUN THE COMPARISON
# ==========================================
# Test how the mini model compares to the industry baseline
inspect_generation("phi4-mini", questions_to_test)
inspect_generation("llama3.1", questions_to_test)

🔍 Initializing the Query Inspector...


🧠 MODEL EVALUATION: PHI4-MINI

🧑‍⚕️ LAYPERSON QUESTION: '"My HbA1c has improved but my sensor shows I am having low blood sugars at night. Which result should I trust?"'
------------------------------------------------------------
  1. HbA1c level improvement with nocturnal hypoglycemia
  2. Discrepancy between glycemic control markers (HbA1c vs fasting glucose)
  3. Nocturnal hypoglycemia in diabetes management assessment
  4. Evaluation of HbA1c versus continuous glucose monitoring for diabetic patients at night
  5. Interpreting conflicting results: HbA1c and overnight blood sugar levels

🧑‍⚕️ PROFESSIONAL QUESTION: 'What are the primary tools used to assess glycemic status in a patient with diabetes?'
------------------------------------------------------------
  1. HbA1c measurement for diabetic control assessment
  2. Fasting plasma glucose test as an indicator of blood sugar levels
  3. Oral Glucose Tolerance Test (OGTT) evaluation method 

### After fixing the parser
Both models produce 5 clean sub-queries. But Llama3.1 still captures more accurate semantics and produces shorter, more concise queries than Phi4-mini. We will stick with Llama3.1 for the following experiment.

In [ ]:
import pandas as pd
from langchain_ollama import ChatOllama
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

PREAMBLE_KEYWORDS = {"here are", "search queries", "following", "below are", "queries:"}

def parse_sub_queries(output):
    lines = [q.strip().strip('"').strip("'").lstrip("1234567890.- ") for q in output.split("\n") if q.strip()]
    return [
        q for q in lines
        if len(q) > 5 and not any(kw in q.lower() for kw in PREAMBLE_KEYWORDS)
    ]

def fuse_document_lists(list_of_doc_lists, rrf_k):
    fused_scores = {}
    for doc_list in list_of_doc_lists:
        for rank, doc in enumerate(doc_list):
            doc_content = doc.page_content
            score = 1.0 / (rrf_k + rank + 1)
            if doc_content in fused_scores:
                fused_scores[doc_content]["score"] += score
            else:
                fused_scores[doc_content] = {"doc_obj": doc, "score": score}
    sorted_fused = sorted(fused_scores.values(), key=lambda x: x["score"], reverse=True)
    return [item["doc_obj"] for item in sorted_fused]

def run_fusion_eval(model_tag, prompt_name, num_queries, rrf_k, eval_df):
    """
    Runs RAG Fusion for each query and evaluates with RAGAS Context Precision.
    Returns DataFrame with per-query scores, sub-queries list, and chunk previews.
    """
    llm = ChatOllama(model=model_tag, temperature=TEMPERATURE, seed=42)
    prompt_template = ChatPromptTemplate.from_template(fusion_prompts[prompt_name])
    chain = prompt_template | llm | StrOutputParser() | parse_sub_queries

    eval_rows = []
    for _, row in eval_df.iterrows():
        question = row['Question']
        golden_ref = row['Reference']
        persona = row['Persona']

        try:
            sub_queries = chain.invoke({
                "num_queries": num_queries,
                "question": question
            })

            search_queries = [question] + sub_queries

            all_retrieved_lists = []
            for sq in search_queries:
                docs = best_hybrid_retriever.invoke(sq)
                all_retrieved_lists.append(docs)

            final_fused_docs = fuse_document_lists(all_retrieved_lists, rrf_k=rrf_k)[:FUSION_TOP_K]
            retrieved_texts = [doc.page_content for doc in final_fused_docs]
            chunk_previews = [doc.page_content.replace('\n', ' ')[:200] for doc in final_fused_docs]

            eval_rows.append({
                "Question": question,
                "Retrieved Contexts": str(retrieved_texts),
                "Reference": str([golden_ref]),
                "Persona": persona,
                "Sub-Queries Generated": len(sub_queries),
                "Sub-Queries": sub_queries,
                "Chunk Previews": chunk_previews
            })
        except Exception as e:
            print(f"    Error on '{question[:50]}...': {e}")
            eval_rows.append({
                "Question": question,
                "Retrieved Contexts": str([]),
                "Reference": str([golden_ref]),
                "Persona": persona,
                "Sub-Queries Generated": 0,
                "Sub-Queries": [],
                "Chunk Previews": []
            })

    results_df = pd.DataFrame(eval_rows)
    print("    Passing to RAGAS LLM Judge...")
    ragas_input = results_df[['Question', 'Retrieved Contexts', 'Reference']].copy()
    scores = run_retrieval(ragas_input)

    if isinstance(scores, pd.DataFrame):
        results_df['context_precision'] = scores['context_precision_C'].values
    else:
        results_df['context_precision'] = scores.values

    return results_df

print("run_fusion_eval defined.")

run_fusion_eval defined.


In [ ]:
prof_sample = retrieved_data_df[retrieved_data_df['Persona'] == 'Professional'].sample(n=5, random_state=42)
layperson_sample = retrieved_data_df[retrieved_data_df['Persona'] == 'Layperson'].sample(n=5, random_state=42)
fusion_eval_df = pd.concat([prof_sample, layperson_sample]).reset_index(drop=True)

print(f"Sampled {len(fusion_eval_df)} evaluation queries:")
for i, row in fusion_eval_df.iterrows():
    print(f"  [{row['Persona'][:4]}] {row['Question'][:80]}...")

Sampled 10 evaluation queries:
  [Prof] What are the primary tools used to assess glycemic status in a patient with diab...
  [Prof] When might a less stringent A1C goal be appropriate?...
  [Prof] Mr. Tan, 52 years old, presents to your clinic after a routine health screening....
  [Prof] A patient with type 2 diabetes on an SGLT2 inhibitor is admitted for elective su...
  [Prof] A newly diagnosed type 2 diabetes patient has CKD with eGFR of 40 mL/min/1.73 m-...
  [Layp] "My HbA1c has improved but my sensor shows I am having low blood sugars at night...
  [Layp] "My doctor wants me to wear a glucose sensor on my arm. What is that and do I re...
  [Layp] "I am going in for surgery next week. Do I need to stop any of my diabetes medic...
  [Layp] "My father is very ill with cancer and diabetes. His sugar levels are not great....
  [Layp] "I cannot afford the glucose sensor. What can I do?"...


In [ ]:
print("=" * 70)
print("PROMPT STRATEGY COMPARISON")
print("Fixed: llama3.1 | N=5 | RRF_K=60")
print("=" * 70)

FIXED_MODEL = "llama3.1"
FIXED_N = 5

step2_results = {}
for prompt_name in ["Multi-Angle", "Clinical_Translation"]:
    print(f"\n  Testing: {prompt_name}...")
    result_df = run_fusion_eval(FIXED_MODEL, prompt_name, FIXED_N, RRF_K, fusion_eval_df)
    step2_results[prompt_name] = result_df
    avg = result_df['context_precision'].mean()
    print(f"    Avg Context Precision: {avg:.4f}")

# ==========================================
# SUMMARY TABLE
# ==========================================
print("\n" + "=" * 70)
print("RESULTS: PROMPT STRATEGY COMPARISON")
print("=" * 70)

step2_summary = []
for name, df in step2_results.items():
    overall = df['context_precision'].mean()
    prof = df[df['Persona'] == 'Professional']['context_precision'].mean()
    lay = df[df['Persona'] == 'Layperson']['context_precision'].mean()
    step2_summary.append({
        "Prompt Strategy": name,
        "Overall": round(overall, 4),
        "Professional": round(prof, 4),
        "Layperson": round(lay, 4)
    })

step2_table = pd.DataFrame(step2_summary).sort_values("Overall", ascending=False)
display(step2_table)

winner_prompt = step2_table.iloc[0]['Prompt Strategy']
loser_prompt = step2_table.iloc[1]['Prompt Strategy']
delta = step2_table.iloc[0]['Overall'] - step2_table.iloc[1]['Overall']
print(f"\nWinner: {winner_prompt} (delta = +{delta:.4f} over {loser_prompt})")

# ==========================================
# PER-QUERY DETAIL (both prompts side-by-side)
# ==========================================
print("\n" + "=" * 70)
print("PER-QUERY CONTEXT PRECISION")
print("=" * 70)

comparison = fusion_eval_df[['Persona', 'Question']].copy()
comparison['Question_Short'] = comparison['Question'].str[:70] + '...'
for pname in ["Multi-Angle", "Clinical_Translation"]:
    comparison[pname] = step2_results[pname]['context_precision'].values
comparison['Delta (MA - CT)'] = comparison['Multi-Angle'] - comparison['Clinical_Translation']
display(comparison[['Persona', 'Question_Short', 'Multi-Angle', 'Clinical_Translation', 'Delta (MA - CT)']])

PROMPT STRATEGY COMPARISON
Fixed: llama3.1 | N=5 | RRF_K=60

  Testing: Multi-Angle...
    Passing to RAGAS LLM Judge...


Evaluating:   0%|          | 0/10 [00:00<?, ?it/s]

    Avg Context Precision: 0.8801

  Testing: Clinical_Translation...
    Passing to RAGAS LLM Judge...


Evaluating:   0%|          | 0/10 [00:00<?, ?it/s]

    Avg Context Precision: 0.8658

RESULTS: PROMPT STRATEGY COMPARISON


,Prompt Strategy,Overall,Professional,Layperson
0,Multi-Angle,0.8801,0.8967,0.8636
1,Clinical_Translation,0.8658,0.9300,0.8017



Winner: Multi-Angle (delta = +0.0143 over Clinical_Translation)

PER-QUERY CONTEXT PRECISION


,Persona,Question_Short,Multi-Angle,Clinical_Translation,Delta (MA - CT)
0,Professional,What are the primary tools used to assess glyc...,0.804167,0.833333,-2.916667e-02
1,Professional,When might a less stringent A1C goal be approp...,1.000000,0.866667,1.333333e-01
2,Professional,"Mr. Tan, 52 years old, presents to your clinic...",0.679167,1.000000,-3.208333e-01
3,Professional,A patient with type 2 diabetes on an SGLT2 inh...,1.000000,1.000000,5.000000e-12
4,Professional,A newly diagnosed type 2 diabetes patient has ...,1.000000,0.950000,5.000000e-02
5,Layperson,"""My HbA1c has improved but my sensor shows I a...",1.000000,0.700000,3.000000e-01
6,Layperson,"""My doctor wants me to wear a glucose sensor o...",1.000000,0.679167,3.208333e-01
7,Layperson,"""I am going in for surgery next week. Do I nee...",0.679167,0.679167,0.000000e+00
8,Layperson,"""My father is very ill with cancer and diabete...",1.000000,1.000000,0.000000e+00
9,Layperson,"""I cannot afford the glucose sensor. What can ...",0.638889,0.950000,-3.111111e-01


In [ ]:
def print_deep_dive(label, row, prompt_name):
    print(f"\n{'='*70}")
    print(f"  [{label}] {prompt_name}")
    print(f"  Persona: {row['Persona']} | Context Precision: {row['context_precision']:.4f}")
    print(f"{'='*70}")
    print(f"\n  ORIGINAL QUESTION:")
    print(f"  {row['Question']}")

    print(f"\n  SUB-QUERIES GENERATED ({len(row['Sub-Queries'])}):")
    for i, sq in enumerate(row['Sub-Queries'], 1):
        print(f"    {i}. {sq}")

    print(f"\n  TOP-{len(row['Chunk Previews'])} FUSED CHUNKS (after RRF merge):")
    for i, chunk in enumerate(row['Chunk Previews'], 1):
        print(f"    Rank {i}: {chunk}...")

    print()


for prompt_name in ["Multi-Angle", "Clinical_Translation"]:
    df = step2_results[prompt_name]

    print("\n" + "#" * 70)
    print(f"#  DEEP DIVE: {prompt_name.upper()}")
    print("#" * 70)

    for persona in ["Professional", "Layperson"]:
        subset = df[df['Persona'] == persona]
        best_idx = subset['context_precision'].idxmax()
        worst_idx = subset['context_precision'].idxmin()

        print_deep_dive(
            f"BEST {persona.upper()}",
            subset.loc[best_idx],
            prompt_name
        )
        print_deep_dive(
            f"WORST {persona.upper()}",
            subset.loc[worst_idx],
            prompt_name
        )


######################################################################
#  DEEP DIVE: MULTI-ANGLE
######################################################################

  [BEST PROFESSIONAL] Multi-Angle
  Persona: Professional | Context Precision: 1.0000

  ORIGINAL QUESTION:
  When might a less stringent A1C goal be appropriate?

  SUB-QUERIES GENERATED (5):
    1. A1C goal in elderly patients with diabetes
    2. Type 2 diabetes A1C target range for frail patients
    3. Less stringent glycemic control in patients with comorbidities
    4. A1C goals in patients with advanced kidney disease
    5. Diabetes management in older adults with multiple chronic conditions

  TOP-5 FUSED CHUNKS (after RRF merge):
    Rank 1:  and stable coexisting chronic illnesses, and intact cognitive and functional status, should have lower glycemic goals (such as A1C <7.0–7.5% [<53–58 mmol/mol]) and/or time in range [TIR] 70–180 mg/dL...
    Rank 2: oals should be individualized and should prioritize avo

### Deep Dive findings

- The query (Mr. Tan, 52yo, new T2DM workup) is simultaneously:

  - Multi-Angle's worst Professional query (0.679)
  - Clinical Translation's best Professional query (1.000)

Multi-Angle's decomposition:

      1. Type 2 diabetes diagnostic criteria fasting glucose
      2. Type 2 diabetes risk factors sedentary lifestyle
      3. Metformin contraindications amlodipine interaction -- hallucinated concern; no clinically significant interaction exists
      4. Hypertension management in type 2 diabetes guidelines
      5. Lifestyle interventions for weight loss in office workers -- drifted to generic wellness
  - Multi-Angle fragmented the vignette into isolated keyword clusters. Sub-query 3 fabricated a drug interaction that doesn't exist, and sub-query 5 drifted into generic lifestyle advice. The resulting chunks are about physical activity for frail elderly (LIFE study) and weight loss pace.

Clinical Translation's decomposition

      1. Type 2 diabetes mellitus diagnosis in a non-diabetic patient with impaired fasting glucose and hemoglobin A1C
      2. Impaired fasting glucose etiology in a sedentary office worker with family history
      3. Hypertension management plan for a patient on amlodipine with new-onset hyperglycemia
      4. Diabetes-related kidney disease screening in a patient with elevated ACR and impaired renal function
      5. Metabolic syndrome diagnosis and management in an overweight office worker

- Clinical Translation understood the vignette as a clinical whole. Each sub-query integrates multiple data points from the case (e.g., sub-query 4 combines the ACR and eGFR values into a proper nephropathy screening query). Sub-query 5 captures the overarching syndrome (BMI 29.5 + hypertension + hyperglycemia = metabolic syndrome). The retrieved chunks include lab test checklists, prediabetes management, hypertension pharmacotherapy, and renal screening.

#### For structured clinical scenarios with multiple interacting data points, Clinical Translation outperforms Multi-Angle fusion prompt.

### Multi-Angle wins overall, but success is split on persona.

- For Professional queries, Clinical Translation fusion prompt performed better. Multi-Angle also performed well enough.

- For Layperson queries, Multi-Angle fusion prompt performed better. While Clinical Translation prompt falls behind.

- Clinical Translation is better when the query already has clean jargon mappings (professional queries and some concrete layperson queries), while Multi-Angle is better when the query is semantically diffuse and needs multiple retrieval angles to converge on the right chunks.

####Multi-Angle wins overall because layperson queries are inherently harder for retrieval (bigger vocabulary gap), so the gains there are larger than the losses on already-easy professional queries.

####Possible future optimization: a persona-aware router that selects the prompt strategy based on detected query complexity. But for a single-prompt choice, Multi-Angle is our safer default.

### Now we have settled on Llama3.1 and Multi-Angle fusion prompt as our best performing config overall. Let's move on to experiment on the number of sub-queries to generate.

In [ ]:
print("=" * 70)
print("NUM_QUERIES (N) COMPARISON")
print("Fixed: llama3.1 | Multi-Angle | RRF_K=60")
print("=" * 70)

FIXED_MODEL = "llama3.1"
FIXED_PROMPT = "Multi-Angle"

step3_results = {}

# Reuse N=5 from Step 2 (identical config)
step3_results["N=5"] = step2_results["Multi-Angle"]
n5_avg = step3_results["N=5"]['context_precision'].mean()
print(f"\n  Reusing N=5 from Step 2: {n5_avg:.4f}")

for n_val in [3, 7]:
    label = f"N={n_val}"
    print(f"\n  Testing: {label}...")
    result_df = run_fusion_eval(FIXED_MODEL, FIXED_PROMPT, n_val, RRF_K, fusion_eval_df)
    step3_results[label] = result_df
    avg = result_df['context_precision'].mean()
    print(f"    Avg Context Precision: {avg:.4f}")

# ==========================================
# SUMMARY TABLE
# ==========================================
print("\n" + "=" * 70)
print("RESULTS: NUM_QUERIES (N) COMPARISON")
print("=" * 70)

step3_summary = []
for label, df in step3_results.items():
    overall = df['context_precision'].mean()
    prof = df[df['Persona'] == 'Professional']['context_precision'].mean()
    lay = df[df['Persona'] == 'Layperson']['context_precision'].mean()
    step3_summary.append({
        "N": label,
        "Overall": round(overall, 4),
        "Professional": round(prof, 4),
        "Layperson": round(lay, 4)
    })

step3_table = pd.DataFrame(step3_summary).sort_values("Overall", ascending=False)
display(step3_table)

winner_n_label = step3_table.iloc[0]['N']
print(f"\nWinner: {winner_n_label}")

# ==========================================
# PER-QUERY SIDE-BY-SIDE
# ==========================================
print("\n" + "=" * 70)
print("PER-QUERY CONTEXT PRECISION")
print("=" * 70)

comparison = fusion_eval_df[['Persona', 'Question']].copy()
comparison['Question_Short'] = comparison['Question'].str[:70] + '...'
for label in ["N=3", "N=5", "N=7"]:
    comparison[label] = step3_results[label]['context_precision'].values
comparison['Delta (N7 - N3)'] = comparison['N=7'] - comparison['N=3']
display(comparison[['Persona', 'Question_Short', 'N=3', 'N=5', 'N=7', 'Delta (N7 - N3)']])

NUM_QUERIES (N) COMPARISON
Fixed: llama3.1 | Multi-Angle | RRF_K=60

  Reusing N=5 from Step 2: 0.8801

  Testing: N=3...
    Passing to RAGAS LLM Judge...


Evaluating:   0%|          | 0/10 [00:00<?, ?it/s]

    Avg Context Precision: 0.8239

  Testing: N=7...


    Passing to RAGAS LLM Judge...


Evaluating:   0%|          | 0/10 [00:00<?, ?it/s]

    Avg Context Precision: 0.8700

RESULTS: NUM_QUERIES (N) COMPARISON


,N,Overall,Professional,Layperson
0,N=5,0.8801,0.8967,0.8636
2,N=7,0.8700,0.8175,0.9225
1,N=3,0.8239,0.8611,0.7867



Winner: N=5

PER-QUERY CONTEXT PRECISION


,Persona,Question_Short,N=3,N=5,N=7,Delta (N7 - N3)
0,Professional,What are the primary tools used to assess glyc...,0.500000,0.804167,0.333333,-0.166667
1,Professional,When might a less stringent A1C goal be approp...,1.000000,1.000000,0.804167,-0.195833
2,Professional,"Mr. Tan, 52 years old, presents to your clinic...",0.805556,0.679167,0.950000,0.144444
3,Professional,A patient with type 2 diabetes on an SGLT2 inh...,1.000000,1.000000,1.000000,0.000000
4,Professional,A newly diagnosed type 2 diabetes patient has ...,1.000000,1.000000,1.000000,0.000000
5,Layperson,"""My HbA1c has improved but my sensor shows I a...",0.804167,1.000000,0.887500,0.083333
6,Layperson,"""My doctor wants me to wear a glucose sensor o...",1.000000,1.000000,0.950000,-0.050000
7,Layperson,"""I am going in for surgery next week. Do I nee...",0.679167,0.679167,0.887500,0.208333
8,Layperson,"""My father is very ill with cancer and diabete...",1.000000,1.000000,1.000000,0.000000
9,Layperson,"""I cannot afford the glucose sensor. What can ...",0.450000,0.638889,0.887500,0.437500


In [ ]:
for n_label in ["N=3", "N=5", "N=7"]:
    df = step3_results[n_label]

    print("\n" + "#" * 70)
    print(f"#  DEEP DIVE: {n_label}")
    print("#" * 70)

    for persona in ["Professional", "Layperson"]:
        subset = df[df['Persona'] == persona]
        best_idx = subset['context_precision'].idxmax()
        worst_idx = subset['context_precision'].idxmin()

        print_deep_dive(
            f"BEST {persona.upper()}",
            subset.loc[best_idx],
            n_label
        )
        print_deep_dive(
            f"WORST {persona.upper()}",
            subset.loc[worst_idx],
            n_label
        )


######################################################################
#  DEEP DIVE: N=3
######################################################################

  [BEST PROFESSIONAL] N=3
  Persona: Professional | Context Precision: 1.0000

  ORIGINAL QUESTION:
  A patient with type 2 diabetes on an SGLT2 inhibitor is admitted for elective surgery. What medication management is required?

  SUB-QUERIES GENERATED (3):
    1. SGLT2 inhibitors perioperative management guidelines
    2. Type 2 diabetes medication management during hospitalization for elective surgery
    3. Perioperative care of patients taking SGLT2 inhibitors and metformin

  TOP-5 FUSED CHUNKS (after RRF merge):
    Rank 1: 0 and 180 mg/dL (5.6 and 10.0 mmol/L). Goals may differ depending on the surgery, risk for hypoglycemia, and glucose-lowering therapy. E SGLT2 Inhibitors in the Perioperative Period SGLT2 inhibitors s...
    Rank 2:  conducted, as an abundance of caution, SGLT2 inhibitors should be held for 3–4 days 

### Findings on different N of subqueries

Professional queries prefer fewer sub-queries. Layperson queries prefer more. This is a similar pattern we observed from testing different fusion prompts.

- Professional queries degrade at N=7
    - The already precise clinical questions come with a narrow correct answer set. At N=7, the model is forced to invent 7 different angles on a narrow scoped question. The extra sub-queries drift into adjacent topics and pull in irrelevant chunks that pollute our RRF ranking. The relevance threshold warnings in our output also confirms this: some sub-queries were so divergent that the semantic retriever returned zero documents above the 0.6 threshold.
- Layperson queries improve at N=7
    - With 7 sub-queries, the retriever casts a wider net to retrieve more potentially relevant chunks about the topics of interest that N=3 and N=5 could have missed. More sub-queries leads to more facets covered.

- The model generates entirely different sets at different N values. The sub-queries are not truncated versions of each other. It doesn't generate N=5 queries as a subset of N=7 queries. We can confirm this by looking at the Mr.Tan query. N=5 is actually the worst for this query. We know that from previous fusion prompt test, the model hallucinated irrelevant sub queries on the Mr.Tan query. In our N queries test, N = 3 actually has a way better score, probably because the three subqueries did not include the hallucinated ones. N = 7 also did well, possibly due to either not hallucinating or overwhelming the hallucinated queries with more valid subqueries.

### N=5 is a balanced configuration. It doesn't win either persona outright, but it avoids the worst failures of both extremes

### Now let's evaluate the final RAG Fusion configs

In [ ]:
# ==========================================
# HOLISTIC RAG FUSION COMPARISON
# 6 Configs x 10 Queries (5 Prof + 5 Lay)
# ==========================================
print("=" * 70)
print("HOLISTIC RAG FUSION EVALUATION")
print("10 queries | llama3.1 | RRF_K=60")
print("=" * 70)

prof_sample = retrieved_data_df[retrieved_data_df['Persona'] == 'Professional'].sample(n=5, random_state=42)
layperson_sample = retrieved_data_df[retrieved_data_df['Persona'] == 'Layperson'].sample(n=5, random_state=42)
fusion_eval_df = pd.concat([prof_sample, layperson_sample]).reset_index(drop=True)

print(f"Evaluation set: {len(fusion_eval_df)} queries (5 Professional, 5 Layperson)")

configs = {
    "MA + N=3": {"prompt": "Multi-Angle",          "n": 3},
    "MA + N=5": {"prompt": "Multi-Angle",          "n": 5},
    "MA + N=7": {"prompt": "Multi-Angle",          "n": 7},
    "CT + N=3": {"prompt": "Clinical_Translation", "n": 3},
    "CT + N=5": {"prompt": "Clinical_Translation", "n": 5},
    "CT + N=7": {"prompt": "Clinical_Translation", "n": 7},
}

holistic_results = {}
for config_name, params in configs.items():
    print(f"\n  Running: {config_name}...")
    result_df = run_fusion_eval(
        model_tag="llama3.1",
        prompt_name=params["prompt"],
        num_queries=params["n"],
        rrf_k=RRF_K,
        eval_df=fusion_eval_df
    )
    holistic_results[config_name] = result_df
    avg = result_df['context_precision'].mean()
    print(f"    Avg Context Precision: {avg:.4f}")

# ==========================================
# SUMMARY TABLE
# ==========================================
print("\n" + "=" * 70)
print("OVERALL LEADERBOARD")
print("=" * 70)

summary_rows = []
for config_name, df in holistic_results.items():
    overall = df['context_precision'].mean()
    prof = df[df['Persona'] == 'Professional']['context_precision'].mean()
    lay = df[df['Persona'] == 'Layperson']['context_precision'].mean()
    summary_rows.append({
        "Config": config_name,
        "Overall": round(overall, 4),
        "Professional": round(prof, 4),
        "Layperson": round(lay, 4),
    })

summary_df = pd.DataFrame(summary_rows).sort_values("Overall", ascending=False)
display(summary_df)

# ==========================================
# PER-PERSONA WINNER CHECK
# ==========================================
print("\n" + "=" * 70)
print("PER-PERSONA BEST CONFIG")
print("=" * 70)

best_prof_config = max(summary_rows, key=lambda x: x["Professional"])
best_lay_config = max(summary_rows, key=lambda x: x["Layperson"])
best_overall_config = max(summary_rows, key=lambda x: x["Overall"])

print(f"  Best Overall:      {best_overall_config['Config']} ({best_overall_config['Overall']:.4f})")
print(f"  Best Professional: {best_prof_config['Config']} ({best_prof_config['Professional']:.4f})")
print(f"  Best Layperson:    {best_lay_config['Config']} ({best_lay_config['Layperson']:.4f})")

routed_score = (best_prof_config['Professional'] + best_lay_config['Layperson']) / 2
print(f"\n  Persona-Routed Score (theoretical): {routed_score:.4f}")
print(f"  Gain over best single config:        +{routed_score - best_overall_config['Overall']:.4f}")

# ==========================================
# PER-QUERY SIDE-BY-SIDE
# ==========================================
print("\n" + "=" * 70)
print("PER-QUERY CONTEXT PRECISION")
print("=" * 70)

detail = fusion_eval_df[['Persona', 'Question']].copy()
detail['Question_Short'] = detail['Question'].str[:70] + '...'
for config_name in configs:
    detail[config_name] = holistic_results[config_name]['context_precision'].values

display(detail[['Persona', 'Question_Short'] + list(configs.keys())])

# ==========================================
# DEEP DIVE: BEST & WORST PER CONFIG
# ==========================================
for config_name in configs:
    df = holistic_results[config_name]

    print("\n" + "#" * 70)
    print(f"#  DEEP DIVE: {config_name}")
    print("#" * 70)

    for persona in ["Professional", "Layperson"]:
        subset = df[df['Persona'] == persona]
        best_idx = subset['context_precision'].idxmax()
        worst_idx = subset['context_precision'].idxmin()

        print_deep_dive(f"BEST {persona.upper()}", subset.loc[best_idx], config_name)
        print_deep_dive(f"WORST {persona.upper()}", subset.loc[worst_idx], config_name)

HOLISTIC RAG FUSION EVALUATION
10 queries | llama3.1 | RRF_K=60
Evaluation set: 10 queries (5 Professional, 5 Layperson)

  Running: MA + N=3...
    Passing to RAGAS LLM Judge...


Evaluating:   0%|          | 0/10 [00:00<?, ?it/s]

    Avg Context Precision: 0.8821

  Running: MA + N=5...
    Passing to RAGAS LLM Judge...


Evaluating:   0%|          | 0/10 [00:00<?, ?it/s]

    Avg Context Precision: 0.7987

  Running: MA + N=7...


...

    Avg Context Precision: 0.8383

OVERALL LEADERBOARD


,Config,Overall,Professional,Layperson
3,CT + N=3,0.9500,1.0000,0.9000
0,MA + N=3,0.8821,0.9133,0.8508
4,CT + N=5,0.8556,0.9286,0.7825
5,CT + N=7,0.8383,0.7900,0.8867
2,MA + N=7,0.8258,0.7967,0.8550
1,MA + N=5,0.7987,0.8092,0.7883



PER-PERSONA BEST CONFIG
  Best Overall:      CT + N=3 (0.9500)
  Best Professional: CT + N=3 (1.0000)
  Best Layperson:    CT + N=3 (0.9000)

  Persona-Routed Score (theoretical): 0.9500
  Gain over best single config:        +0.0000

PER-QUERY CONTEXT PRECISION


,Persona,Question_Short,MA + N=3,MA + N=5,MA + N=7,CT + N=3,CT + N=5,CT + N=7
0,Professional,What are the primary tools used to assess glyc...,0.700000,0.366667,0.416667,1.0,0.755556,0.500000
1,Professional,When might a less stringent A1C goal be approp...,0.866667,1.000000,1.000000,1.0,0.887500,0.450000
2,Professional,"Mr. Tan, 52 years old, presents to your clinic...",1.000000,0.679167,0.679167,1.0,1.000000,1.000000
3,Professional,A patient with type 2 diabetes on an SGLT2 inh...,1.000000,1.000000,1.000000,1.0,1.000000,1.000000
4,Professional,A newly diagnosed type 2 diabetes patient has ...,1.000000,1.000000,0.887500,1.0,1.000000,1.000000
5,Layperson,"""My HbA1c has improved but my sensor shows I a...",0.950000,1.000000,0.887500,1.0,0.866667,1.000000
6,Layperson,"""My doctor wants me to wear a glucose sensor o...",1.000000,0.679167,1.000000,1.0,1.000000,1.000000
7,Layperson,"""I am going in for surgery next week. Do I nee...",0.804167,0.679167,0.804167,1.0,0.679167,0.679167
8,Layperson,"""My father is very ill with cancer and diabete...",1.000000,1.000000,1.000000,1.0,1.000000,0.804167
9,Layperson,"""I cannot afford the glucose sensor. What can ...",0.500000,0.583333,0.583333,0.5,0.366667,0.950000



######################################################################
#  DEEP DIVE: MA + N=3
######################################################################

  [BEST PROFESSIONAL] MA + N=3
  Persona: Professional | Context Precision: 1.0000

  ORIGINAL QUESTION:
  Mr. Tan, 52 years old, presents to your clinic after a routine health screening. He was told his blood sugar was high and was referred for further assessment. He has no prior diabetes diagnosis. He reports increased thirst and frequent urination over the past 2 months. He works as an office manager, is sedentary, and eats out frequently. He has hypertension managed with amlodipine. No known drug allergies. Family history of type 2 diabetes in his father. Vitals: BP 138/86 mmHg, HR 78 bpm, BMI 29.5 kg/m-square. Labs: Fasting glucose 11.2 mmol/L, A1C 9.2%, eGFR 72 mL/min/1.73 m-square, urine albumin-to-creatinine ratio (ACR) 18 mg/g (normal). Lipid panel pending. What is your SOAP assessment and management plan?

  SUB

### Final configs evaluation
1. N=3 beats N=5 and N=7 for both prompts. Fewer sub-queries helps in eliminating the potential hallucinations, consistently producing better retrieval.
2. Clinical Translation beats Multi-Angle at every N value with combined configurations.


Earlier when we compared the fusion prompts alone, we thought Professional prefers Clinical Translation, while Layperson prefers Multi-Angle. This is no longer valid after we combine multiple configurations.

#### The best config without doubt is Llama3.1 + Clinical Translation + N=3


# HyDE — Hypothetical Document Embeddings

HyDE is fundamentally different from RAG Fusion. Instead of generating new *questions*, it asks the LLM to generate a **hypothetical answer** (a "hallucinated" document). The embeddings of that hypothetical answer are then used to find real documents in the VDB that are mathematically similar.


We will use Llama3.1 and a dedicated prompt for generating answers in the same clinical style with the ADA documents.

In [ ]:
import pandas as pd
from langchain_ollama import ChatOllama
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

HYDE_PROMPT = """You are a senior physician writing a clinical reference passage.
Write a concise passage (3-5 sentences) in the style of a formal clinical guideline
to answer the following medical question.
Do not say you are generating a hypothetical document. Write the passage directly.

Question: {question}
Passage:"""

HYDE_TOP_K = 5

def run_hyde_eval(model_tag, retriever, eval_df):
    """
    Runs HyDE retrieval for each query and evaluates with RAGAS Context Precision.
    1. Generate hypothetical clinical answer
    2. Use hypothetical answer as query for the given retriever
    3. Score with run_retrieval
    """
    llm = ChatOllama(model=model_tag, temperature=0.1, seed=42)
    prompt_template = ChatPromptTemplate.from_template(HYDE_PROMPT)
    hyde_chain = prompt_template | llm | StrOutputParser()

    eval_rows = []
    for _, row in eval_df.iterrows():
        question = row['Question']
        golden_ref = row['Reference']
        persona = row['Persona']

        try:
            hypothetical_doc = hyde_chain.invoke({"question": question})

            docs = retriever.invoke(hypothetical_doc)[:HYDE_TOP_K]
            retrieved_texts = [doc.page_content for doc in docs]
            chunk_previews = [doc.page_content.replace('\n', ' ')[:200] for doc in docs]

            eval_rows.append({
                "Question": question,
                "Retrieved Contexts": str(retrieved_texts),
                "Reference": str([golden_ref]),
                "Persona": persona,
                "Hypothetical Doc": hypothetical_doc,
                "Chunk Previews": chunk_previews
            })
        except Exception as e:
            print(f"    Error on '{question[:50]}...': {e}")
            eval_rows.append({
                "Question": question,
                "Retrieved Contexts": str([]),
                "Reference": str([golden_ref]),
                "Persona": persona,
                "Hypothetical Doc": "",
                "Chunk Previews": []
            })

    results_df = pd.DataFrame(eval_rows)
    print("    Passing to RAGAS LLM Judge...")
    ragas_input = results_df[['Question', 'Retrieved Contexts', 'Reference']].copy()
    scores = run_retrieval(ragas_input)

    if isinstance(scores, pd.DataFrame):
        results_df['context_precision'] = scores['context_precision_C'].values
    else:
        results_df['context_precision'] = scores.values

    return results_df

print("run_hyde_eval defined.")

run_hyde_eval defined.


In [ ]:
print("=" * 70)
print("HyDE EVALUATION: RETRIEVER COMPARISON")
print("llama3.1 | Clinical Persona | seed=42")
print("=" * 70)

prof_sample = retrieved_data_df[retrieved_data_df['Persona'] == 'Professional'].sample(n=5, random_state=42)
layperson_sample = retrieved_data_df[retrieved_data_df['Persona'] == 'Layperson'].sample(n=5, random_state=42)
hyde_eval_df = pd.concat([prof_sample, layperson_sample]).reset_index(drop=True)

print(f"Evaluation set: {len(hyde_eval_df)} queries (5 Professional, 5 Layperson)\n")

hyde_configs = {
    "HyDE + Semantic": semantic_retriever,
    "HyDE + Hybrid Vector Dominant (80/20)": best_hybrid_retriever,
}

hyde_results = {}
for config_name, retriever in hyde_configs.items():
    print(f"  Running: {config_name}...")
    result_df = run_hyde_eval("llama3.1", retriever, hyde_eval_df)
    hyde_results[config_name] = result_df
    avg = result_df['context_precision'].mean()
    print(f"    Avg Context Precision: {avg:.4f}")

# ==========================================
# SUMMARY TABLE
# ==========================================
print("\n" + "=" * 70)
print("HyDE RESULTS: RETRIEVER COMPARISON")
print("=" * 70)

hyde_summary = []
for config_name, df in hyde_results.items():
    overall = df['context_precision'].mean()
    prof = df[df['Persona'] == 'Professional']['context_precision'].mean()
    lay = df[df['Persona'] == 'Layperson']['context_precision'].mean()
    hyde_summary.append({
        "Config": config_name,
        "Overall": round(overall, 4),
        "Professional": round(prof, 4),
        "Layperson": round(lay, 4),
    })

hyde_table = pd.DataFrame(hyde_summary).sort_values("Overall", ascending=False)
display(hyde_table)

# ==========================================
# PER-QUERY SIDE-BY-SIDE
# ==========================================
print("\n" + "=" * 70)
print("PER-QUERY CONTEXT PRECISION")
print("=" * 70)

comparison = hyde_eval_df[['Persona', 'Question']].copy()
comparison['Question_Short'] = comparison['Question'].str[:70] + '...'
for config_name in hyde_configs:
    comparison[config_name] = hyde_results[config_name]['context_precision'].values
comparison['Delta (Hybrid - Semantic)'] = (
    comparison['HyDE + Hybrid Vector Dominant (80/20)'] - comparison['HyDE + Semantic']
)
display(comparison[['Persona', 'Question_Short', 'HyDE + Semantic', 'HyDE + Hybrid Vector Dominant (80/20)', 'Delta (Hybrid - Semantic)']])

# ==========================================
# DEEP DIVE: BEST & WORST PER CONFIG
# ==========================================
def print_hyde_deep_dive(label, row, config_name):
    print(f"\n{'='*70}")
    print(f"  [{label}] {config_name}")
    print(f"  Persona: {row['Persona']} | Context Precision: {row['context_precision']:.4f}")
    print(f"{'='*70}")
    print(f"\n  ORIGINAL QUESTION:")
    print(f"  {row['Question']}")

    print(f"\n  HYPOTHETICAL DOCUMENT (generated by LLM):")
    print(f"  {row['Hypothetical Doc']}")

    print(f"\n  TOP-{len(row['Chunk Previews'])} RETRIEVED CHUNKS:")
    for i, chunk in enumerate(row['Chunk Previews'], 1):
        print(f"    Rank {i}: {chunk}...")

    print()


for config_name in hyde_configs:
    df = hyde_results[config_name]

    print("\n" + "#" * 70)
    print(f"#  DEEP DIVE: {config_name}")
    print("#" * 70)

    for persona in ["Professional", "Layperson"]:
        subset = df[df['Persona'] == persona]
        best_idx = subset['context_precision'].idxmax()
        worst_idx = subset['context_precision'].idxmin()

        print_hyde_deep_dive(f"BEST {persona.upper()}", subset.loc[best_idx], config_name)
        print_hyde_deep_dive(f"WORST {persona.upper()}", subset.loc[worst_idx], config_name)

HyDE EVALUATION: RETRIEVER COMPARISON
llama3.1 | Clinical Persona | seed=42
Evaluation set: 10 queries (5 Professional, 5 Layperson)

  Running: HyDE + Semantic...
    Passing to RAGAS LLM Judge...


Evaluating:   0%|          | 0/10 [00:00<?, ?it/s]

    Avg Context Precision: 0.8871
  Running: HyDE + Hybrid Vector Dominant (80/20)...
    Passing to RAGAS LLM Judge...


Evaluating:   0%|          | 0/10 [00:00<?, ?it/s]

    Avg Context Precision: 0.9483

HyDE RESULTS: RETRIEVER COMPARISON


,Config,Overall,Professional,Layperson
1,HyDE + Hybrid Vector Dominant (80/20),0.9483,0.9567,0.94
0,HyDE + Semantic,0.8871,0.8342,0.94



PER-QUERY CONTEXT PRECISION


,Persona,Question_Short,HyDE + Semantic,HyDE + Hybrid Vector Dominant (80/20),Delta (Hybrid - Semantic)
0,Professional,What are the primary tools used to assess glyc...,0.366667,0.833333,0.466667
1,Professional,When might a less stringent A1C goal be approp...,0.804167,0.950000,0.145833
2,Professional,"Mr. Tan, 52 years old, presents to your clinic...",1.000000,1.000000,0.000000
3,Professional,A patient with type 2 diabetes on an SGLT2 inh...,1.000000,1.000000,0.000000
4,Professional,A newly diagnosed type 2 diabetes patient has ...,1.000000,1.000000,0.000000
5,Layperson,"""My HbA1c has improved but my sensor shows I a...",1.000000,1.000000,0.000000
6,Layperson,"""My doctor wants me to wear a glucose sensor o...",1.000000,1.000000,0.000000
7,Layperson,"""I am going in for surgery next week. Do I nee...",1.000000,1.000000,0.000000
8,Layperson,"""My father is very ill with cancer and diabete...",1.000000,1.000000,0.000000
9,Layperson,"""I cannot afford the glucose sensor. What can ...",0.700000,0.700000,0.000000



######################################################################
#  DEEP DIVE: HyDE + Semantic
######################################################################

  [BEST PROFESSIONAL] HyDE + Semantic
  Persona: Professional | Context Precision: 1.0000

  ORIGINAL QUESTION:
  A patient with type 2 diabetes on an SGLT2 inhibitor is admitted for elective surgery. What medication management is required?

  HYPOTHETICAL DOCUMENT (generated by LLM):
  For patients with type 2 diabetes on sodium-glucose cotransporter-2 (SGLT2) inhibitors who are undergoing elective surgery, the medication should be held on the day of surgery and continued as soon as possible post-operatively, provided that renal function has returned to baseline. This approach minimizes the risk of acute kidney injury associated with SGLT2 inhibitor use in the perioperative period. If an SGLT2 inhibitor cannot be restarted within 24 hours after surgery, consideration should be given to temporarily switching to a d

### HyDE Results
HyDE + Hybrid Vector Dominant (80/20) beats HyDE + Semantic Search.

# Full Evaluation on all candidates

**5 Retrieval Configs x 97 Queries in the Golden Eval Set | Full metric suite via `run_full_evaluation`**

| # | Config | Retriever | LLM-Augmented? |
|---|--------|-----------|----------------|
| 1 | Semantic Baseline (Threshold=0.6) | Vector only | No |
| 2 | Semantic (MMR) | Vector only (diversity) | No |
| 3 | Hybrid 80/20 | Semantic Baseline + BM25 (k1=0.5, b=0.75) | No |
| 4 | RAG Fusion (CT + N=3) | Hybrid 80/20 + RRF | Yes (sub-queries) |
| 5 | HyDE + Hybrid 80/20 | Semantic + BM25 via hypothetical doc | Yes (hypothetical doc) |

In [ ]:
import os
import re
import shutil
import pandas as pd
import chromadb
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma
from langchain_community.retrievers import BM25Retriever
from langchain_classic.retrievers.ensemble import EnsembleRetriever

# ==========================================
# 1. EMBEDDING MODEL + CHROMA VECTOR STORE
# ==========================================
embedding = HuggingFaceEmbeddings(model_name="BAAI/bge-small-en-v1.5")

drive_path = '/content/drive/MyDrive/Colab Notebooks/GenAI_Grp/ADA_Diabetes_db_20260312'
local_path = '/content/local_diabetes_db'

if not os.path.exists(local_path):
    shutil.copytree(drive_path, local_path, dirs_exist_ok=True)
    print(f"Copied Chroma DB to {local_path}")
else:
    print(f"Using existing local copy at {local_path}")

client = chromadb.PersistentClient(path=local_path)
collection = client.get_collection("ADA_Diabetes_db_20260312")

vectorstore = Chroma(
    collection_name='ADA_Diabetes_db_20260312',
    embedding_function=embedding,
    client=client
)
print(f"Vector store loaded: {collection.count()} documents")

# ==========================================
# 2. BM25 RETRIEVER
# ==========================================
all_data = collection.get(include=['documents', 'metadatas'])
all_doc_texts = all_data['documents']
all_doc_metadatas = all_data['metadatas']

STOP_WORDS = {"what", "are", "the", "of", "a", "an", "is", "was", "were",
              "and", "or", "but", "in", "on", "at", "to", "for", "with",
              "it", "its", "kind", "can", "i", "ive", "been", "having",
              "too", "this", "do", "my", "so", "get", "about", "how", "why"}

def ultimate_tokenize(text):
    text = text.lower()
    text = re.sub(r'[-/:]', ' ', text)
    raw_tokens = text.split()
    final_tokens = []
    for token in raw_tokens:
        clean_token = token.strip(',;!?()"\'[]{}')
        if clean_token.endswith('.') and clean_token.count('.') == 1:
            clean_token = clean_token[:-1]
        if clean_token and clean_token not in STOP_WORDS:
            final_tokens.append(clean_token)
    return final_tokens

bm25_retriever = BM25Retriever.from_texts(
    texts=all_doc_texts,
    metadatas=all_doc_metadatas,
    preprocess_func=ultimate_tokenize,
    bm25_params={"k1": 0.5, "b": 0.75}
)
bm25_retriever.k = 5
print(f"BM25 retriever ready ({len(all_doc_texts)} documents)")

# ==========================================
# 3. SEMANTIC RETRIEVER (base for hybrid ensemble)
# ==========================================
semantic_retriever = vectorstore.as_retriever(
    search_type="similarity_score_threshold",
    search_kwargs={"score_threshold": 0.6, "k": 5}
)

# ==========================================
# 4. HYBRID RETRIEVERS
# ==========================================
weight_configs = {
    "1. Vector Dominant (80/20)": [0.8, 0.2],
    "2. Vector Leaning (60/40)": [0.6, 0.4],
    "3. Balanced (50/50)": [0.5, 0.5],
    "4. Keyword Leaning (40/60)": [0.4, 0.6],
    "5. Keyword Dominant (20/80)": [0.2, 0.8],
}

hybrid_retrievers = {}
for config_name, weights in weight_configs.items():
    hybrid_retrievers[config_name] = EnsembleRetriever(
        retrievers=[semantic_retriever, bm25_retriever],
        weights=weights
    )

best_hybrid_retriever = hybrid_retrievers["1. Vector Dominant (80/20)"]
print(f"Hybrid retrievers ready ({len(hybrid_retrievers)} configs)")

# ==========================================
# 5. RAG FUSION PROMPTS & HELPERS
# ==========================================
PREAMBLE_KEYWORDS = {"here are", "search queries", "following", "below are", "queries:"}

def parse_sub_queries(output):
    lines = [q.strip().strip('"').strip("'").lstrip("1234567890.- ") for q in output.split("\n") if q.strip()]
    return [
        q for q in lines
        if len(q) > 5 and not any(kw in q.lower() for kw in PREAMBLE_KEYWORDS)
    ]

def fuse_document_lists(list_of_doc_lists, rrf_k):
    fused_scores = {}
    for doc_list in list_of_doc_lists:
        for rank, doc in enumerate(doc_list):
            doc_content = doc.page_content
            score = 1.0 / (rrf_k + rank + 1)
            if doc_content in fused_scores:
                fused_scores[doc_content]["score"] += score
            else:
                fused_scores[doc_content] = {"doc_obj": doc, "score": score}
    sorted_fused = sorted(fused_scores.values(), key=lambda x: x["score"], reverse=True)
    return [item["doc_obj"] for item in sorted_fused]

fusion_prompts = {
    "Multi-Angle": """You are an expert clinical research assistant. Your goal is to retrieve relevant medical guidelines to answer the user's question.
Generate exactly {num_queries} distinct search queries that explore this topic from different clinical domains (e.g., diagnostic criteria, pharmacological management, pathophysiology, lifestyle interventions).

CRITICAL INSTRUCTIONS:
- Write concise, keyword-rich search phrases (e.g., "type 2 diabetes metformin contraindications").
- DO NOT write full interrogative questions (Avoid "What are the...").
- Output ONLY the raw queries, one per line. Absolutely NO numbers, NO bullet points, NO quotes, and NO introductory text.

Original Question: {question}""",

    "Clinical_Translation": """You are a senior medical terminologist. Translate the user's question into exactly {num_queries} distinct, highly precise clinical search queries.

CRITICAL INSTRUCTIONS:
- Convert layman terminology into standard clinical jargon (e.g., "weak heart" -> "congestive heart failure etiology").
- Stick to terminology used in standard medical guidelines and broad pharmacological classes. Avoid ultra-obscure jargon.
- Write concise, keyword-dense search phrases, NOT conversational sentences.
- Output ONLY the raw queries, one per line. Absolutely NO numbers, NO bullet points, NO quotes, and NO conversational filler.

Original Question: {question}"""
}

RRF_K = 60
FUSION_TOP_K = 5
TEMPERATURE = 0.1

# ==========================================
# 6. GOLDEN EVAL SET
# ==========================================
file_path = "/content/drive/MyDrive/Colab Notebooks/GenAI_Grp/Eval Results.xlsx"
sheet_name = "ada_mistral"

retrieved_data_df = pd.read_excel(file_path, sheet_name=sheet_name)
print(f"\nGolden eval set loaded: {len(retrieved_data_df)} rows (sheet: {sheet_name})")
print(f"Columns: {list(retrieved_data_df.columns)}")

In [ ]:
import pandas as pd
import time
from langchain_ollama import ChatOllama
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

# ==========================================
# 1. SHARED SETUP
# ==========================================
ANSWER_PROMPT = ChatPromptTemplate.from_template(
 """
You are a helpful medical assistant. Use the following pieces of retrieved context to answer the question.
If you don't know the answer, just say that you don't know. Do not make up information.

Context:
{context}

Question:
{question}

Answer:
"""
)

answer_llm = ChatOllama(model="mistral:7b", temperature=0, seed=42)
answer_chain = ANSWER_PROMPT | answer_llm | StrOutputParser()

HYDE_PROMPT_TEXT = """You are a senior physician writing a clinical reference passage.
Write a concise passage (3-5 sentences) in the style of a formal clinical guideline
to answer the following medical question.
Do not say you are generating a hypothetical document. Write the passage directly.

Question: {question}
Passage:"""

def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

# ==========================================
# 2. FULL GOLDEN EVAL SET
# ==========================================
eval_df = retrieved_data_df.reset_index(drop=True)
n_prof = len(eval_df[eval_df['Persona'] == 'Professional'])
n_lay = len(eval_df[eval_df['Persona'] == 'Layperson'])
print(f"Evaluation set: {len(eval_df)} queries ({n_prof} Professional, {n_lay} Layperson)")

# ==========================================
# 3. DEFINE RETRIEVERS
# ==========================================
semantic_threshold_ret = vectorstore.as_retriever(
    search_type="similarity_score_threshold",
    search_kwargs={"score_threshold": 0.6, "k": 5}
)

mmr_ret = vectorstore.as_retriever(
    search_type="mmr",
    search_kwargs={"k": 5, "fetch_k": 20, "lambda_mult": 0.5}
)

hybrid_80_20 = hybrid_retrievers["1. Vector Dominant (80/20)"]

# ==========================================
# 4. PIPELINE FUNCTIONS
# ==========================================
def run_simple_pipeline(retriever, config_name):
    """Configs 1-3: retrieve with original query, then generate answer."""
    print(f"\n  Running: {config_name}...")
    rows = []
    for _, row in eval_df.iterrows():
        question = row['Question']
        start = time.time()

        docs = retriever.invoke(question)[:5]
        context = format_docs(docs)
        answer = answer_chain.invoke({"context": context, "question": question})
        latency = time.time() - start

        rows.append({
            "Persona": row['Persona'],
            "Question": question,
            "Golden Answer": row['Golden Answer'],
            "Reference": row['Reference'],
            "Retrieved Contexts": str([doc.page_content for doc in docs]),
            "Generated Answer": answer,
            "Latency Seconds": round(latency, 3),
        })
    print(f"    Retrieval + Generation done ({len(rows)} queries).")
    return pd.DataFrame(rows)


def run_fusion_pipeline():
    """Config 4: RAG Fusion (CT + N=3) via Hybrid 80/20, then generate answer."""
    print(f"\n  Running: RAG Fusion (CT + N=3)...")
    fusion_llm = ChatOllama(model="llama3.1", temperature=0.1, seed=42)
    ct_prompt = ChatPromptTemplate.from_template(fusion_prompts["Clinical_Translation"])
    fusion_chain = ct_prompt | fusion_llm | StrOutputParser() | parse_sub_queries

    rows = []
    for _, row in eval_df.iterrows():
        question = row['Question']
        start = time.time()

        sub_queries = fusion_chain.invoke({"num_queries": 3, "question": question})
        search_queries = [question] + sub_queries

        all_retrieved_lists = []
        for sq in search_queries:
            docs = best_hybrid_retriever.invoke(sq)
            all_retrieved_lists.append(docs)

        final_docs = fuse_document_lists(all_retrieved_lists, rrf_k=60)[:5]
        context = format_docs(final_docs)
        answer = answer_chain.invoke({"context": context, "question": question})
        latency = time.time() - start

        rows.append({
            "Persona": row['Persona'],
            "Question": question,
            "Golden Answer": row['Golden Answer'],
            "Reference": row['Reference'],
            "Retrieved Contexts": str([doc.page_content for doc in final_docs]),
            "Generated Answer": answer,
            "Latency Seconds": round(latency, 3),
        })
    print(f"    Retrieval + Generation done ({len(rows)} queries).")
    return pd.DataFrame(rows)


def run_hyde_pipeline(retriever):
    """Config 5: HyDE (Clinical) + retriever, then generate answer."""
    print(f"\n  Running: HyDE + Hybrid 80/20...")
    hyde_llm = ChatOllama(model="llama3.1", temperature=0.1, seed=42)
    hyde_prompt = ChatPromptTemplate.from_template(HYDE_PROMPT_TEXT)
    hyde_chain = hyde_prompt | hyde_llm | StrOutputParser()

    rows = []
    for _, row in eval_df.iterrows():
        question = row['Question']
        start = time.time()

        hypothetical_doc = hyde_chain.invoke({"question": question})
        docs = retriever.invoke(hypothetical_doc)[:5]
        context = format_docs(docs)
        answer = answer_chain.invoke({"context": context, "question": question})
        latency = time.time() - start

        rows.append({
            "Persona": row['Persona'],
            "Question": question,
            "Golden Answer": row['Golden Answer'],
            "Reference": row['Reference'],
            "Retrieved Contexts": str([doc.page_content for doc in docs]),
            "Generated Answer": answer,
            "Latency Seconds": round(latency, 3),
        })
    print(f"    Retrieval + Generation done ({len(rows)} queries).")
    return pd.DataFrame(rows)


EVAL_OUTPUT_FILE = "final_evaluation_results.xlsx"

def save_eval_sheet(df, sheet_name):
    _mode = 'a' if os.path.exists(EVAL_OUTPUT_FILE) else 'w'
    _kw = {"mode": _mode, "engine": "openpyxl"}
    if _mode == 'a':
        _kw["if_sheet_exists"] = "replace"
    with pd.ExcelWriter(EVAL_OUTPUT_FILE, **_kw) as writer:
        df.to_excel(writer, sheet_name=sheet_name[:31], index=False)

print("Shared setup complete.")

Evaluation set: 97 queries (61 Professional, 36 Layperson)
Shared setup complete.


In [ ]:
config_name = "1. Semantic (Threshold=0.6)"
print(f"Running pipeline: {config_name}...")
pipeline_df = run_simple_pipeline(semantic_threshold_ret, config_name)
print(f"Running evaluation: {config_name}...")
evaluated_df = run_full_evaluation(pipeline_df)
save_eval_sheet(evaluated_df, config_name)
print(f"Saved to {EVAL_OUTPUT_FILE}")

Running pipeline: 1. Semantic (Threshold=0.6)...

  Running: 1. Semantic (Threshold=0.6)...


    Retrieval + Generation done (97 queries).
Running evaluation: 1. Semantic (Threshold=0.6)...
1. Calculating Deterministic Metrics (Readability)...
2. Calculating Custom Persona Metrics (Lexical Adaptation, Safety, Tone, Structural Usability with Reasonings)...
3. Calculating Ragas
3. Running Ragas Core Metrics (Context Utilization, Faithfulness, Answer Correctness, Context Precision)


Evaluating:   0%|          | 0/388 [00:00<?, ?it/s]

4. Calculating Retrieval
Running Pure Retrieval Metrics 'Context Precision' comparing golden_set Reference vs Retrievec Contexts


Evaluating:   0%|          | 0/97 [00:00<?, ?it/s]

Saved to final_evaluation_results.xlsx


In [ ]:
config_name = "2. Semantic (MMR)"
print(f"Running pipeline: {config_name}...")
pipeline_df = run_simple_pipeline(mmr_ret, config_name)
print(f"Running evaluation: {config_name}...")
evaluated_df = run_full_evaluation(pipeline_df)
save_eval_sheet(evaluated_df, config_name)
print(f"Saved to {EVAL_OUTPUT_FILE}")

Running pipeline: 2. Semantic (MMR)...

  Running: 2. Semantic (MMR)...
    Retrieval + Generation done (97 queries).
Running evaluation: 2. Semantic (MMR)...
1. Calculating Deterministic Metrics (Readability)...
2. Calculating Custom Persona Metrics (Lexical Adaptation, Safety, Tone, Structural Usability with Reasonings)...
3. Calculating Ragas
3. Running Ragas Core Metrics (Context Utilization, Faithfulness, Answer Correctness, Context Precision)


Evaluating:   0%|          | 0/388 [00:00<?, ?it/s]

ERROR:ragas.executor:Exception raised in Job[131]: BadRequestError(Error code: 400 - {'error': {'message': "We could not parse the JSON body of your request. (HINT: This likely means you aren't using your HTTP library correctly. The OpenAI API expects a JSON payload, but what was sent was not valid JSON. If you have trouble figuring out how to fix this, please contact us through our help center at help.openai.com.)", 'type': 'invalid_request_error', 'param': None, 'code': None}})
ERROR:ragas.executor:Exception raised in Job[286]: BadRequestError(Error code: 400 - {'error': {'message': "We could not parse the JSON body of your request. (HINT: This likely means you aren't using your HTTP library correctly. The OpenAI API expects a JSON payload, but what was sent was not valid JSON. If you have trouble figuring out how to fix this, please contact us through our help center at help.openai.com.)", 'type': 'invalid_request_error', 'param': None, 'code': None}})
ERROR:ragas.executor:Exception

4. Calculating Retrieval
Running Pure Retrieval Metrics 'Context Precision' comparing golden_set Reference vs Retrievec Contexts


Evaluating:   0%|          | 0/97 [00:00<?, ?it/s]

Saved to final_evaluation_results.xlsx


In [ ]:
config_name = "3. Hybrid 80/20"
print(f"Running pipeline: {config_name}...")
pipeline_df = run_simple_pipeline(hybrid_80_20, config_name)
print(f"Running evaluation: {config_name}...")
evaluated_df = run_full_evaluation(pipeline_df)
save_eval_sheet(evaluated_df, config_name)
print(f"Saved to {EVAL_OUTPUT_FILE}")

Running pipeline: 3. Hybrid 80/20...

  Running: 3. Hybrid 80/20...


    Retrieval + Generation done (97 queries).
Running evaluation: 3. Hybrid 80/20...
1. Calculating Deterministic Metrics (Readability)...
2. Calculating Custom Persona Metrics (Lexical Adaptation, Safety, Tone, Structural Usability with Reasonings)...
3. Calculating Ragas
3. Running Ragas Core Metrics (Context Utilization, Faithfulness, Answer Correctness, Context Precision)


Evaluating:   0%|          | 0/388 [00:00<?, ?it/s]

ERROR:ragas.executor:Exception raised in Job[122]: BadRequestError(Error code: 400 - {'error': {'message': "We could not parse the JSON body of your request. (HINT: This likely means you aren't using your HTTP library correctly. The OpenAI API expects a JSON payload, but what was sent was not valid JSON. If you have trouble figuring out how to fix this, please contact us through our help center at help.openai.com.)", 'type': 'invalid_request_error', 'param': None, 'code': None}})


In [ ]:
config_name = "4. RAG Fusion (CT+N=3)"
print(f"Running pipeline: {config_name}...")
pipeline_df = run_fusion_pipeline()
print(f"Running evaluation: {config_name}...")
evaluated_df = run_full_evaluation(pipeline_df)
save_eval_sheet(evaluated_df, config_name)
print(f"Saved to {EVAL_OUTPUT_FILE}")

In [ ]:
config_name = "5. HyDE + Hybrid 80/20"
print(f"Running pipeline: {config_name}...")
pipeline_df = run_hyde_pipeline(hybrid_80_20)
print(f"Running evaluation: {config_name}...")
evaluated_df = run_full_evaluation(pipeline_df)
save_eval_sheet(evaluated_df, config_name)
print(f"Saved to {EVAL_OUTPUT_FILE}")

In [ ]:
import os
import pandas as pd

EVAL_OUTPUT_FILE = "final_evaluation_results.xlsx"

SHEET_NAMES = [
    "1. Semantic (Threshold=0.6)",
    "2. Semantic (MMR)",
    "3. Hybrid 80/20",
    "4. RAG Fusion (CT+N=3)",
    "5. HyDE + Hybrid 80/20",
]

METRIC_COLS = [
    'context_precision_C', 'context_utilization', 'faithfulness',
    'answer_correctness', 'readability',
    'lexical_adaptation_score', 'safety_triage_score',
    'tone_alignment_score', 'structural_usability_score'
]

evaluated_results = {}
for name in SHEET_NAMES:
    evaluated_results[name] = pd.read_excel(EVAL_OUTPUT_FILE, sheet_name=name[:31])

print("=" * 70)
print("FINAL LEADERBOARD — ALL METRICS")
print("=" * 70)

leaderboard_rows = []
for config_name, df in evaluated_results.items():
    row = {"Config": config_name}
    for col in METRIC_COLS:
        if col in df.columns:
            row[col] = round(df[col].mean(), 4)
    row["Avg Latency (s)"] = round(df["Latency Seconds"].mean(), 2)
    leaderboard_rows.append(row)

leaderboard = pd.DataFrame(leaderboard_rows)
display(leaderboard.sort_values("context_precision_C", ascending=False))

print("\n" + "=" * 70)
print("PER-PERSONA BREAKDOWN")
print("=" * 70)

KEY_METRICS = ['context_precision_C', 'faithfulness', 'answer_correctness']

for persona in ["Professional", "Layperson"]:
    print(f"\n--- {persona} ---")
    persona_rows = []
    for config_name, df in evaluated_results.items():
        subset = df[df['Persona'] == persona]
        row = {"Config": config_name}
        for col in KEY_METRICS:
            if col in subset.columns:
                row[col] = round(subset[col].mean(), 4)
        persona_rows.append(row)
    display(pd.DataFrame(persona_rows).sort_values("context_precision_C", ascending=False))

with pd.ExcelWriter(EVAL_OUTPUT_FILE, mode='a', engine='openpyxl', if_sheet_exists='replace') as writer:
    leaderboard.to_excel(writer, sheet_name="Leaderboard", index=False)

print(f"\nLeaderboard saved to {EVAL_OUTPUT_FILE}")